# QSVM on WDBC

This notebook is organized around three goals: classical baselines, simulation code, and result exploration for the WDBC QSVC/VQC study.

Note: the real-hardware workflow lives in `REAL_QPU.ipynb`. This notebook stays focused on simulations, visualizations, and analysis.

## Reader guide

Use the notebook in this order:

1. **Data loading and quick visualization**: loads the WDBC dataset and gives the first sanity-check plots.
2. **Classical baselines**: rebuilds or validates the classical RFE, logistic-regression, and SVM reference results used later.
3. **Simulation code**: contains the main scaled QSVC+VQC sweep and the no-scaling QSVC sweep. These are the cells that generate the quantum result CSV files.
4. **Simulation result exploration**: compares classical vs quantum results and builds the main accuracy, AUC, heatmap, ROC/PR, and error-profile figures.
5. **Geometry and feature-space exploration**: collects the sphere visualizations and feature-selection follow-up for the best-performing configurations.

Reference notes:

- The main simulation blocks are explicitly labeled with markdown right above them.
- Most analysis cells expect the CSV outputs from the simulation cells to already exist.
- `REAL_QPU.ipynb` is the separate notebook for hardware execution and should be read independently from this one.
- If the goal is only to reproduce figures from saved results, the simulation cells do not need to be rerun.

## Data loading and quick visualization

In [ ]:
# as kernel we will be working with the lab as backend  
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load the Wisconsin Breast Cancer dataset
data = load_breast_cancer()

# Create a DataFrame with the dataset
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target  # Add target (diagnosis) column

# Display ALL THE ROW FOR EXPLORATION 

df.head()

# Describe the first few columns
# df.iloc[:, :].describe()


In [ ]:
import matplotlib.pyplot as plt

# Scatter plot for two features
plt.figure(figsize=(8, 6))
plt.scatter(df['mean radius'], df['mean texture'], c=df['target'], cmap='coolwarm', alpha=0.7)
plt.colorbar(label='Target (0=Benign, 1=Malignant)')
plt.xlabel('Mean Radius')
plt.ylabel('Mean Texture')
plt.title('Scatter Plot of Mean Radius vs. Mean Texture')
plt.show()


In [ ]:
import seaborn as sns

# Select a subset of features
features = ['mean radius', 'mean texture', 'mean perimeter', 'mean area', 'mean smoothness']
pair_data = df[features + ['target']]

# Pair plot
sns.pairplot(pair_data, hue='target', diag_kind='kde', palette='coolwarm')
plt.show()


In [ ]:
# Compute correlation matrix
corr_matrix = df.corr()

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True)
plt.title('Correlation Heatmap')
plt.show()


In [ ]:
# Plot histogram
plt.figure(figsize=(8, 6))
plt.hist(df['mean radius'], bins=30, color='blue', alpha=0.7)
plt.xlabel('Mean Radius')
plt.ylabel('Frequency')
plt.title('Distribution of Mean Radius')
plt.show()


In [ ]:
# Box plot
sns.boxplot(x='target', y='mean area', data=df, palette='coolwarm')
plt.xlabel('Target (0=Benign, 1=Malignant)')
plt.ylabel('Mean Area')
plt.title('Box Plot of Mean Area by Target')
plt.show()


## Classical baselines

These cells rebuild or validate the classical reference tables used later by the simulation analysis.

In [ ]:
import pandas as pd
from sklearn.datasets          import load_breast_cancer
from sklearn.metrics           import accuracy_score

from fixed_split_preprocessing import fixed_split, prepare_features, rfe_estimator

# 1) Load data
data  = load_breast_cancer()
X_raw = pd.DataFrame(data.data, columns=data.feature_names)
y     = data.target

# 2) Split raw rows before any learned preprocessing
X_tr, X_te, y_tr, y_te = fixed_split(X_raw, y)
scalers = ("Standardization", "Normalization")
estimators = ("Logistic Regression", "Random Forest")
feature_counts = [4, 7, 11, 17, 18]

# 3) Fit each preprocessing configuration on the training partition
results = []
for scale_name in scalers:
    for est_name in estimators:
        for k in feature_counts:
            X_tr_sel, X_te_sel, features = prepare_features(
                X_tr, X_te, y_tr, scale_name, est_name, k
            )
            est = rfe_estimator(est_name)
            # train & evaluate
            est.fit(X_tr_sel, y_tr)
            train_acc = accuracy_score(y_tr, est.predict(X_tr_sel))
            test_acc  = accuracy_score(y_te, est.predict(X_te_sel))
            overfit   = train_acc - test_acc
            
            # record
            results.append({
                "Scaling":           scale_name,
                "Estimator":         est_name,
                "n_features":        k,
                "Selected Features": features,
                "Train Accuracy":    train_acc,
                "Test Accuracy":     test_acc,
                "Overfit Score":     overfit
            })

# 4) Build DataFrame and display/save
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Optionally save:
results_df.to_csv('classical_rfe_fixed_counts_train_only.csv', index=False)



In [ ]:
import ast
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

CONFIG_CSV = "classical_rfe_fixed_counts_train_only.csv"

# Fixed train/test split (this reproduces your original experiments)
TEST_SIZE = 0.20
RANDOM_STATE = 42

# ====== LOAD DATASET (WDBC) ======
data = load_breast_cancer()
X_full = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")


def build_scaler(name: str):
    """Return a scaler instance based on the 'Scaling' column."""
    if not isinstance(name, str):
        return None
    name_lower = name.lower()
    if name_lower.startswith("standard"):
        return StandardScaler()
    if "minmax" in name_lower or "min-max" in name_lower or "normalization" in name_lower:
        return MinMaxScaler()
    return None  # no scaling


def build_estimator(name: str):
    """Return a classifier instance based on the 'Estimator' column."""
    if not isinstance(name, str):
        raise ValueError("Estimator name must be a string")
    name_lower = name.lower()
    if "logistic" in name_lower:
        # Matches your original LR configs (C=1.0, lbfgs, max_iter=1000)
        return LogisticRegression(
            C=1.0,
            max_iter=1000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )
    if "random forest" in name_lower:
        # Matches your RF configs (n_estimators=100, default settings)
        return RandomForestClassifier(
            n_estimators=100,
            random_state=RANDOM_STATE,
        )
    raise ValueError(f"Unknown Estimator: {name}")


def evaluate_config(row):
    """Re-train and evaluate one config row, return metrics."""
    # Parse selected features (string → Python list)
    features_str = row["Selected Features"]
    features = ast.literal_eval(features_str)

    # Subset data to those features
    X_sel = X_full[features].copy()

    # Train/test split (same for all configs)
    X_train, X_test, y_train, y_test = train_test_split(
        X_sel,
        y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    # Build scaler & estimator
    scaler = build_scaler(row["Scaling"])
    clf = build_estimator(row["Estimator"])

    # Apply scaling if needed
    if scaler is not None:
        X_train_proc = scaler.fit_transform(X_train)
        X_test_proc = scaler.transform(X_test)
    else:
        X_train_proc = X_train.values
        X_test_proc = X_test.values

    # Fit model
    clf.fit(X_train_proc, y_train)

    # Predictions
    y_train_pred = clf.predict(X_train_proc)
    y_test_pred = clf.predict(X_test_proc)

    # Probabilities (for ROC AUC)
    if hasattr(clf, "predict_proba"):
        y_test_proba = clf.predict_proba(X_test_proc)[:, 1]
    else:
        # Fallback: use decision_function if available
        if hasattr(clf, "decision_function"):
            y_test_proba = clf.decision_function(X_test_proc)
        else:
            y_test_proba = None

    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    overfit = train_acc - test_acc
    f1 = f1_score(y_test, y_test_pred)

    if y_test_proba is not None:
        try:
            auc = roc_auc_score(y_test, y_test_proba)
        except Exception:
            auc = np.nan
    else:
        auc = np.nan

    return {
        "Recalc_Train_Acc": train_acc,
        "Recalc_Test_Acc": test_acc,
        "Recalc_Overfit": overfit,
        "Recalc_Test_ROC_AUC": auc,
        "Recalc_F1": f1,
    }


def main():
    df_cfg = pd.read_csv(CONFIG_CSV)

    # Recalculate metrics for each config
    recalculated = []
    for idx, row in df_cfg.iterrows():
        metrics = evaluate_config(row)
        recalculated.append(metrics)

    rec_df = pd.DataFrame(recalculated)
    out = pd.concat([df_cfg, rec_df], axis=1)

    # Optional: check differences between original and recalculated accuracy
    if "Train Accuracy" in df_cfg.columns and "Test Accuracy" in df_cfg.columns:
        out["Delta_Train_Acc"] = out["Recalc_Train_Acc"] - out["Train Accuracy"]
        out["Delta_Test_Acc"] = out["Recalc_Test_Acc"] - out["Test Accuracy"]

    # Save to CSV with AUC filled
    out_file = "classical_rfe_configs_with_auc_train_only.csv"
    out.to_csv(out_file, index=False)
    print(f"Saved recalculated metrics (including Test ROC_AUC) to: {out_file}")

    # Quick preview
    out


if __name__ == "__main__":
    main()


In [ ]:
import ast
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    confusion_matrix,
    f1_score,
)

# ------------------------------------------------------------------
# 1. Paths / config
# ------------------------------------------------------------------
CSV_PATH = "CLASSICAL_svm_on_rfe_configs.csv"

# ------------------------------------------------------------------
# 2. Load CSV with SVM configs
# ------------------------------------------------------------------
# Use the tracked table for its configuration keys only.
archived_configs = pd.read_csv(CSV_PATH)[
    ["Scaling", "Orig_Estimator", "SVM_Model", "n_features"]
]
fitted_features = pd.read_csv("classical_rfe_fixed_counts_train_only.csv").rename(
    columns={"Estimator": "Orig_Estimator"}
)
keys = ["Scaling", "Orig_Estimator", "n_features"]
df = archived_configs.merge(
    fitted_features[keys + ["Selected Features"]], on=keys, how="left",
    validate="many_to_one",
)
if df["Selected Features"].isna().any():
    raise ValueError("Missing training-only feature selection for an SVM configuration")

# ------------------------------------------------------------------
# 3. Load WDBC data and rebuild the same split
# ------------------------------------------------------------------
data = load_breast_cancer()
X_full = data.data
y_full = data.target          # 0 = malignant, 1 = benign
feature_names = list(data.feature_names)

# Same split as used originally (this reproduces your Test Accuracy & AUC)
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_full,
    y_full,
    test_size=0.2,
    random_state=42,
    stratify=y_full,
)

# ------------------------------------------------------------------
# 4. Helpers
# ------------------------------------------------------------------
def make_scaler(scaling_name: str):
    """
    Map your 'Scaling' column to a scikit-learn scaler.
    - 'Standardization' (or containing 'standard') -> StandardScaler
    - everything else -> MinMaxScaler (Normalization)
    """
    s = (scaling_name or "").lower()
    if "standard" in s or "z-score" in s or "zscore" in s:
        return StandardScaler()
    else:
        return MinMaxScaler()

def eval_config(row):
    """
    Re-train SVM for one row of the CSV and compute:
    - accuracy, AUC (for sanity check)
    - F1 (malignant), Sensitivity (malignant), Specificity (benign)
    - TP, FP, TN, FN  (positive class = malignant = 0)
    """
    # --- 1) subset features according to "Selected Features"
    sel_feats = ast.literal_eval(row["Selected Features"])
    idx = [feature_names.index(f) for f in sel_feats]

    X_tr = X_train_full[:, idx]
    X_te = X_test_full[:, idx]

    # --- 2) scaling
    scaler = make_scaler(row["Scaling"])
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    # --- 3) SVM model (matches SVM_Model column: RBF, C=1.0, gamma='scale')
    clf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
    clf.fit(X_tr_s, y_train)

    # --- 4) predictions & probabilities
    y_pred = clf.predict(X_te_s)
    y_prob = clf.predict_proba(X_te_s)[:, 1]  # prob of class 1 (benign), for ROC-AUC

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    # --- 5) confusion matrix (labels=[0,1], positive = malignant = 0)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tp, fn = cm[0, 0], cm[0, 1]   # malignant
    fp, tn = cm[1, 0], cm[1, 1]   # benign

    sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")  # recall malignant
    spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")  # recall benign
    f1   = f1_score(y_test, y_pred, pos_label=0)              # F1 for malignant

    return acc, auc, f1, sens, spec, tp, fp, tn, fn

# ------------------------------------------------------------------
# 5. Loop over all rows and compute the missing metrics
# ------------------------------------------------------------------
acc_list, auc_list = [], []
f1_list, sens_list, spec_list = [], [], []
tp_list, fp_list, tn_list, fn_list = [], [], [], []

for _, row in df.iterrows():
    acc, auc, f1, sens, spec, tp, fp, tn, fn = eval_config(row)

    # Optional sanity check: these should match existing columns
    # print(row["n_features"], acc, row["Test Accuracy"], auc, row["Test ROC_AUC"])

    acc_list.append(acc)
    auc_list.append(auc)
    f1_list.append(f1 * 100.0)      # convert to %
    sens_list.append(sens * 100.0)  # %
    spec_list.append(spec * 100.0)  # %
    tp_list.append(tp)
    fp_list.append(fp)
    tn_list.append(tn)
    fn_list.append(fn)

# ------------------------------------------------------------------
# 6. Attach new columns and save back to the same CSV
# ------------------------------------------------------------------
df["Test Accuracy"] = acc_list
df["Test ROC_AUC"] = auc_list
df["F1_malignant_%"]        = f1_list
df["Sensitivity_malignant_%"] = sens_list
df["Specificity_benign_%"]  = spec_list
df["TP_malignant"]          = tp_list
df["FP_malignant"]          = fp_list
df["TN_benign"]             = tn_list
df["FN_malignant"]          = fn_list

out_file = "classical_svm_on_rfe_configs_train_only.csv"
df.to_csv(out_file, index=False)
print("Updated file written to:", out_file)


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# File paths (adapt if needed)
# --------------------------------------------------
LR_CSV  = "classical_rfe_configs_with_auc_checked.csv"
SVM_CSV = "CLASSICAL_svm_on_rfe_configs.csv"


# --------------------------------------------------
# Helpers
# --------------------------------------------------
def _fmt_pct(x, digits=2):
    try:
        return f"{float(x)*100:.{digits}f}"
    except Exception:
        return "NA"

def _fmt_str(x):
    return "NA" if pd.isna(x) else str(x)


# --------------------------------------------------
# Logistic Regression: best per k and global best
# --------------------------------------------------
def best_lr_configs(lr_csv: str):
    df = pd.read_csv(lr_csv)

    # Keep only Logistic Regression configs
    df_lr = df[df["Estimator"].str.contains("Logistic", case=False, na=False)].copy()

    # Safety: ensure metric columns exist
    for col in ["Recalc_Test_Acc", "Recalc_Test_ROC_AUC", "Recalc_F1"]:
        if col not in df_lr.columns:
            df_lr[col] = np.nan

    # Best per k
    rows = []
    for k, g in df_lr.groupby("n_features"):
        g2 = g.sort_values(
            ["Recalc_Test_Acc", "Recalc_Test_ROC_AUC", "Recalc_F1"],
            ascending=[False, False, False],
            kind="mergesort",
        )
        rows.append(g2.iloc[0])
    best_per_k = pd.DataFrame(rows).sort_values("n_features").reset_index(drop=True)

    # Global best (by accuracy, then AUC, then F1)
    global_best = best_per_k.sort_values(
        ["Recalc_Test_Acc", "Recalc_Test_ROC_AUC", "Recalc_F1"],
        ascending=[False, False, False],
        kind="mergesort",
    ).iloc[0]

    return best_per_k, global_best


# --------------------------------------------------
# SVM: best per k and global best
# --------------------------------------------------
def best_svm_configs(svm_csv: str):
    df = pd.read_csv(svm_csv)

    # Restrict to RFE(LogReg) case
    df_svm = df[df["Orig_Estimator"].str.contains("Logistic", case=False, na=False)].copy()

    # Safety
    for col in ["Test Accuracy", "Test ROC_AUC"]:
        if col not in df_svm.columns:
            df_svm[col] = np.nan

    # Best per k
    rows = []
    for k, g in df_svm.groupby("n_features"):
        g2 = g.sort_values(
            ["Test Accuracy", "Test ROC_AUC"],
            ascending=[False, False],
            kind="mergesort",
        )
        rows.append(g2.iloc[0])
    best_per_k = pd.DataFrame(rows).sort_values("n_features").reset_index(drop=True)

    # Global best (by AUC, then accuracy)
    global_best = best_per_k.sort_values(
        ["Test ROC_AUC", "Test Accuracy"],
        ascending=[False, False],
        kind="mergesort",
    ).iloc[0]

    return best_per_k, global_best


# --------------------------------------------------
# Main
# --------------------------------------------------
if __name__ == "__main__":
    # --- Logistic Regression ---
    lr_best_k, lr_best_global = best_lr_configs(LR_CSV)

    print("=== Logistic Regression: best per k (RFE(LogReg)) ===")
    for _, r in lr_best_k.iterrows():
        k   = int(r["n_features"])
        scl = _fmt_str(r["Scaling"])
        acc = _fmt_pct(r["Recalc_Test_Acc"])
        auc = _fmt_pct(r["Recalc_Test_ROC_AUC"])
        f1  = _fmt_pct(r["Recalc_F1"])
        print(f"k={k:2d} | Scaling={scl:>14} | Test_Acc={acc}% | AUC={auc}% | F1={f1}%")

    print("\n=== Logistic Regression: GLOBAL BEST (by Test_Acc, then AUC, F1) ===")
    k   = int(lr_best_global["n_features"])
    scl = _fmt_str(lr_best_global["Scaling"])
    acc = _fmt_pct(lr_best_global["Recalc_Test_Acc"])
    auc = _fmt_pct(lr_best_global["Recalc_Test_ROC_AUC"])
    f1  = _fmt_pct(lr_best_global["Recalc_F1"])
    print(
        "CLASSICAL_LR_BEST | "
        f"Model=LogisticRegression, k={k}, Scaling={scl}, "
        f"Test_Acc={acc}%, AUC={auc}%, F1={f1}%"
    )

    # --- SVM ---
    svm_best_k, svm_best_global = best_svm_configs(SVM_CSV)

    print("\n=== SVM (RBF, RFE(LogReg)): best per k ===")
    for _, r in svm_best_k.iterrows():
        k   = int(r["n_features"])
        scl = _fmt_str(r["Scaling"])
        acc = _fmt_pct(r["Test Accuracy"])
        auc = _fmt_pct(r["Test ROC_AUC"])
        print(f"k={k:2d} | Scaling={scl:>14} | Test_Acc={acc}% | AUC={auc}%")

    print("\n=== SVM: GLOBAL BEST (by AUC, then Test_Acc) ===")
    k   = int(svm_best_global["n_features"])
    scl = _fmt_str(svm_best_global["Scaling"])
    acc = _fmt_pct(svm_best_global["Test Accuracy"])
    auc = _fmt_pct(svm_best_global["Test ROC_AUC"])
    model_name = _fmt_str(svm_best_global.get("SVM_Model", "SVM"))
    print(
        "CLASSICAL_SVM_BEST | "
        f"Model={model_name}, k={k}, Scaling={scl}, "
        f"Test_Acc={acc}%, AUC={auc}%"
    )


In [ ]:
import pandas as pd

# 1. Load the CSV
df = pd.read_csv('classical_rfe_fixed_counts.csv')

# 2. For each n_features, select the row with highest Test Accuracy
feature_counts = [4, 7, 11, 17, 18]
df_sub = df[df['n_features'].isin(feature_counts)]
best_rows = df_sub.loc[df_sub.groupby('n_features')['Test Accuracy'].idxmax()]

# 3. Build Markdown table
columns = ['n_features', 'Scaling', 'Estimator', 'Test Accuracy', 'Overfit Score']
md_lines = []
md_lines.append("| n_features | Scaling | Estimator | Test Accuracy | Overfit Score |")
md_lines.append("|------------|---------|-----------|---------------|---------------|")
for _, row in best_rows.iterrows():
    md_lines.append(
        f"| {int(row['n_features'])} | {row['Scaling']} | {row['Estimator']} "
        f"| {row['Test Accuracy']:.4f} | {row['Overfit Score']:.4f} |"
    )
md_table = "\n".join(md_lines)

# 4. Print and save
print(md_table)
with open('best_classical_table.md', 'w') as f:
    f.write(md_table)


## Simulation code

The next cells are the main simulation blocks used to generate the scaled and no-scaling experiment tables.

The sweep cells refit scaling and RFE using only the fixed training partition. Their train-only CSVs are separate from the archived results. Scores still describe one fixed split; use training-only model selection for held-out comparisons.

### Simulation code: scaled QSVC + VQC sweep

In [ ]:
# filename: quantum_sweep.py
import os
import time
import traceback

import pandas as pd
from tqdm.auto import tqdm

from sklearn.datasets        import load_breast_cancer
from fixed_split_preprocessing import fixed_split, prepare_features
from sklearn.metrics         import accuracy_score, roc_auc_score, f1_score, confusion_matrix

from qiskit.circuit.library import ZZFeatureMap, ZFeatureMap, PauliFeatureMap, RealAmplitudes
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms.classifiers import QSVC, VQC

# COBYLA moved around across Qiskit versions; these fallbacks help.
try:
    from qiskit_algorithms.optimizers import COBYLA  # preferred in newer stacks
except Exception:
    try:
        from qiskit.algorithms.optimizers import COBYLA  # older qiskit
    except Exception:
        from qiskit_machine_learning.optimizers import COBYLA  # if present as last resort

from qiskit.primitives import StatevectorSampler


def run_job(scale_name, estimator, k, fmap_name, ent, reps,
            X_tr_sel, X_te_sel, y_tr, y_te, fmap_cache):
    t0 = time.time()

    # b) lookup cached feature map
    fmap = fmap_cache[(k, fmap_name, ent, reps)]

    # c) QSVC
    qk   = FidelityStatevectorKernel(feature_map=fmap)
    qsvc = QSVC(quantum_kernel=qk, probability=True)
    qsvc.fit(X_tr_sel, y_tr)
    y_tr_q, y_te_q = qsvc.predict(X_tr_sel), qsvc.predict(X_te_sel)
    q_train = accuracy_score(y_tr, y_tr_q)
    q_test  = accuracy_score(y_te, y_te_q)
    q_over  = q_train - q_test
    q_auc   = roc_auc_score(y_te, qsvc.predict_proba(X_te_sel)[:, 1])
    q_f1    = f1_score(y_te, y_te_q)
    tn, fp, fn, tp = confusion_matrix(y_te, y_te_q).ravel()

    # d) VQC
    ansatz = RealAmplitudes(num_qubits=k, reps=1)
    vqc    = VQC(
        feature_map=fmap,
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=100),
        sampler=StatevectorSampler()
    )
    vqc.fit(X_tr_sel, y_tr)
    y_tr_v, y_te_v = vqc.predict(X_tr_sel), vqc.predict(X_te_sel)
    v_train = accuracy_score(y_tr, y_tr_v)
    v_test  = accuracy_score(y_te, y_te_v)
    v_over  = v_train - v_test
    v_auc   = roc_auc_score(y_te, vqc.predict_proba(X_te_sel)[:, 1])
    v_f1    = f1_score(y_te, y_te_v)
    tn2, fp2, fn2, tp2 = confusion_matrix(y_te, y_te_v).ravel()

    duration = time.time() - t0

    return {
        'Scaling': scale_name, 'Estimator': estimator, 'n_features': k,
        'Feature_Map': fmap_name, 'Entanglement': ent, 'Reps': reps,
        'QSVC_Train_Acc': round(q_train,4), 'QSVC_Test_Acc': round(q_test,4),
        'QSVC_Overfit': round(q_over,4), 'QSVC_AUC': round(q_auc,4),
        'QSVC_F1': round(q_f1,4), 'QSVC_TP': int(tp), 'QSVC_FP': int(fp),
        'QSVC_TN': int(tn), 'QSVC_FN': int(fn),
        'VQC_Train_Acc': round(v_train,4), 'VQC_Test_Acc': round(v_test,4),
        'VQC_Overfit': round(v_over,4), 'VQC_AUC': round(v_auc,4),
        'VQC_F1': round(v_f1,4), 'VQC_TP': int(tp2), 'VQC_FP': int(fp2),
        'VQC_TN': int(tn2), 'VQC_FN': int(fn2),
        'Duration_sec': round(duration,4)
    }


def safe_run_job(*args, **kwargs):
    try:
        return run_job(*args, **kwargs)
    except Exception:
        print("❌ Exception in job:", args[:6])
        traceback.print_exc()
        return None


if __name__ == "__main__":
    # Read configuration keys only. Refit all feature lists below.
    results_df = pd.read_csv('classical_rfe_fixed_counts.csv')[
        ['Scaling', 'Estimator', 'n_features']
    ]
    results_df = results_df[results_df['n_features'].isin([4, 7, 11, 17])]

    data = load_breast_cancer()
    X_raw = pd.DataFrame(data.data, columns=data.feature_names)
    X_tr, X_te, y_tr, y_te = fixed_split(X_raw, data.target)

    # Feature-map factories
    feature_map_builders = {
        'ZZFeatureMap':    lambda d, r, e: ZZFeatureMap(feature_dimension=d, reps=r, entanglement=e),
        'ZFeatureMap':     lambda d, r, e: ZFeatureMap(feature_dimension=d, reps=r),  # entanglement ignored by design
        'PauliFeatureMap': lambda d, r, e: PauliFeatureMap(feature_dimension=d, reps=r, entanglement=e),
    }
    entanglements = ['full', 'linear', 'circular']
    reps_list     = [1, 2, 3, 4]

    # 3) CSV setup
    out_csv = 'quantum_each_results_train_only.csv'
    header = [
        'Scaling','Estimator','n_features',
        'Feature_Map','Entanglement','Reps',
        'QSVC_Train_Acc','QSVC_Test_Acc','QSVC_Overfit',
        'QSVC_AUC','QSVC_F1','QSVC_TP','QSVC_FP','QSVC_TN','QSVC_FN',
        'VQC_Train_Acc','VQC_Test_Acc','VQC_Overfit',
        'VQC_AUC','VQC_F1','VQC_TP','VQC_FP','VQC_TN','VQC_FN',
        'Duration_sec'
    ]
    if not os.path.exists(out_csv):
        pd.DataFrame(columns=header).to_csv(out_csv, index=False)

    df_done      = pd.read_csv(out_csv)
    num_existing = len(df_done)
    num_total    = (
        len(results_df)
        * len(feature_map_builders)
        * len(entanglements)
        * len(reps_list)
    )
    print(f"🔧 CPU cores available: {os.cpu_count()}")
    print(f"📄 '{out_csv}' has {num_existing} rows already.")
    print(f"🔢 Total configuration keys: {num_total}\n")

    # 4) Build the cache once
    fmap_cache = {}
    for _, row in results_df.iterrows():
        k = int(row['n_features'])
        for fmap_name, builder in feature_map_builders.items():
            for ent in entanglements:
                for reps in reps_list:
                    key = (k, fmap_name, ent, reps)
                    fmap_cache[key] = builder(d=k, r=reps, e=ent)

    # 5) Build jobs skipping existing
    existing = set(zip(
        df_done['Scaling'], df_done['Estimator'], df_done['n_features'],
        df_done['Feature_Map'], df_done['Entanglement'], df_done['Reps']
    ))
    jobs = []
    for _, row in results_df.iterrows():
        scale = row['Scaling']
        est = row['Estimator']
        k = int(row['n_features'])
        pending = []
        for fmap_name in feature_map_builders:
            for ent in entanglements:
                for reps in reps_list:
                    key = (scale, est, k, fmap_name, ent, reps)
                    if key not in existing:
                        pending.append((scale, est, k, fmap_name, ent, reps))
        if pending:
            train_selected, test_selected, _ = prepare_features(X_tr, X_te, y_tr, scale, est, k)
            jobs.extend((*key, train_selected, test_selected) for key in pending)

    print(f"🚀 {len(jobs)} new jobs to run.\n")

    # 6) Sequential with a tqdm bar, appending each result immediately
    for job in tqdm(jobs, desc="Quantum sweep"):
        res = safe_run_job(*job,
                           y_tr=y_tr,
                           y_te=y_te,
                           fmap_cache=fmap_cache)
        if res is not None:
            pd.DataFrame([res]).to_csv(out_csv, mode='a', header=False, index=False)

    print("\n✅ All done — results in", out_csv)


### Simulation code: no-scaling QSVC sweep

In [ ]:
# Use the same implementation as the standalone sweep.
%run quantum_no_scaling_sweep.py


## Simulation result exploration

These cells analyze the CSV outputs from the simulation sweeps and generate the main comparison figures.

The following analysis cells refer to the historical result files and configuration manifests. The plots describe archived results. Their input files are separate from the train-only outputs above.

In [ ]:
# Accuracy vs Qubits (k) — Classical vs Quantum (scaled vs no-scaling)
# Classical: classical_rfe_fixed_counts.csv  (uses: n_features, Test Accuracy)
# Quantum (scaled): quantum_each_results.csv  (uses: QSVC_Test_Acc or similar)
# Quantum (no scaling): quantum_no_scaling_qsvc_results.csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- USER PATHS ----------------
CLASSICAL_CSV = "classical_rfe_fixed_counts.csv"
Q_SCALED_CSV  = "quantum_each_results.csv"
Q_NOSCALE_CSV = "quantum_no_scaling_qsvc_results.csv"
KS = [4, 7, 11, 17]

# Curve look
SMOOTH = True                 # smooth curves (interpolate + light moving average)
SMOOTH_DENSE_POINTS = 80      # more points → smoother lines
SMOOTH_WINDOW = 10             # odd integer; 3–7 is usually good
BASELINE = 0.90               # area fill baseline (y-value)
MARKER_SIZE = 6
LINE_WIDTH = 2.2
FILL_ALPHA = 0.10
# --------------------------------------------

def parse_pct_like(x):
    """Return numeric accuracy in [0,1] from 0.9825, 98.25, or '98.25%'."""
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)):
        x = float(x)
        return x/100.0 if x > 1.0 else x
    s = str(x).strip()
    if s.endswith("%"): s = s[:-1]
    try:
        v = float(s)
        return v/100.0 if v > 1.0 else v
    except:
        return np.nan

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def best_acc_by_k(df, k_col, acc_col, ks):
    df2 = df.copy()
    df2["_k"]   = pd.to_numeric(df2[k_col], errors="coerce").astype("Int64")
    df2["_acc"] = df2[acc_col].map(parse_pct_like)
    df2 = df2.dropna(subset=["_k","_acc"])
    out = {}
    for k in ks:
        sub = df2[df2["_k"] == k]
        if not sub.empty:
            out[k] = float(sub["_acc"].max())
    return out

# ----------- LOADERS (headers adapted to your files) -----------
def load_classical_best(csv_path, ks):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().replace("\n"," ") for c in df.columns]
    kcol   = pick_col(df, ["n_features", "k", "Features", "Features (→ qubits)", "n_feats"])
    acccol = pick_col(df, ["Test Accuracy", "Test Accuracy (%)", "Test_Acc", "Accuracy", "ACC", "Acc"])
    if kcol is None or acccol is None:
        raise ValueError("Classical CSV must contain 'n_features' and 'Test Accuracy' (or equivalent).")
    return best_acc_by_k(df, kcol, acccol, ks)

def load_quantum_best(csv_path, ks):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().replace("\n"," ") for c in df.columns]
    kcol   = pick_col(df, ["n_features","k","Features"])
    acccol = pick_col(df, ["QSVC_Test_Acc","Test_Acc","Accuracy","ACC"])
    if kcol is None or acccol is None:
        raise ValueError(f"Quantum CSV ({csv_path}) needs 'n_features' and a test accuracy column.")
    return best_acc_by_k(df, kcol, acccol, ks)

def moving_average(y, window):
    if window <= 1: return y
    w = int(window)
    if w % 2 == 0: w += 1  # ensure odd
    pad = w // 2
    ypad = np.pad(y, (pad,pad), mode="edge")
    kernel = np.ones(w)/w
    return np.convolve(ypad, kernel, mode="valid")

def smooth_curve(xs, ys, dense_points=80, window=5):
    xs, ys = np.asarray(xs, float), np.asarray(ys, float)
    m = np.isfinite(xs) & np.isfinite(ys)
    xs, ys = xs[m], ys[m]
    if len(xs) < 2: return xs, ys
    xd = np.linspace(xs.min(), xs.max(), dense_points)
    yd = np.interp(xd, xs, ys)
    yd = moving_average(yd, window)
    return xd, yd

# ------------- Load data -------------
best_classical = load_classical_best(CLASSICAL_CSV, KS)
best_q_scaled  = load_quantum_best(Q_SCALED_CSV,  KS)
best_q_noscal  = load_quantum_best(Q_NOSCALE_CSV, KS)

# Align x-axis
xs = [k for k in KS if (k in best_classical) or (k in best_q_scaled) or (k in best_q_noscal)]
yc = [best_classical.get(k, np.nan) for k in xs]
yq = [best_q_scaled.get(k, np.nan)  for k in xs]
yn = [best_q_noscal.get(k, np.nan)  for k in xs]

# ------------- Plot -------------
plt.figure(figsize=(8,5), dpi=130)

def plot_series_with_fill(xs, ys, label, linestyle, marker):
    xs = np.asarray(xs, float)
    ys = np.asarray(ys, float)

    # markers at true k points
    plt.plot(xs, ys, linestyle="none", marker=marker, markersize=MARKER_SIZE, label=None)

    # smooth vs straight line
    if SMOOTH:
        xd, yd = smooth_curve(xs, ys, dense_points=SMOOTH_DENSE_POINTS, window=SMOOTH_WINDOW)
        plt.plot(xd, yd, linestyle, linewidth=LINE_WIDTH, label=label)
        # area fill
        plt.fill_between(xd, BASELINE, yd, alpha=FILL_ALPHA)
    else:
        plt.plot(xs, ys, linestyle, linewidth=LINE_WIDTH, label=label)
        plt.fill_between(xs, BASELINE, ys, alpha=FILL_ALPHA)

# Classical
plot_series_with_fill(xs, yc, "Classical", "-",  "o")
# Quantum + Scaling
plot_series_with_fill(xs, yq, "Quantum + Scaling", "--", "s")
# Quantum no Scaling
plot_series_with_fill(xs, yn, "Quantum no Scaling", ":", "^")

plt.ylim(0.90, 1.00)
plt.xlim(min(xs)-0.5, max(xs)+0.5)
plt.xticks(xs)
plt.ylabel("Test Accuracy")
plt.xlabel("Number of Qubits (n_features)")
plt.title("Test Accuracy vs. Qubit Count: Classical vs. Quantum (Scaled vs. Unscaled)")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

# -------------------------------------------------
# 1. Helper: convert [0,1] → [%] if needed
# -------------------------------------------------
def percentize(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().le(1.0 + 1e-12).mean() >= 0.95:
        return s * 100.0
    return s

# -------------------------------------------------
# 2. Load CSVs
# -------------------------------------------------
df_classical = pd.read_csv("classical_rfe_configs_with_auc_checked.csv")
df_q_scaled  = pd.read_csv("quantum_each_results.csv")
df_q_noscale = pd.read_csv("quantum_no_scaling_qsvc_results.csv")

# Columns with AUC
df_classical["Recalc_Test_ROC_AUC"] = percentize(df_classical["Recalc_Test_ROC_AUC"])
df_q_scaled["QSVC_AUC"]             = percentize(df_q_scaled["QSVC_AUC"])
df_q_noscale["QSVC_AUC"]           = percentize(df_q_noscale["QSVC_AUC"])

# -------------------------------------------------
# 3. Feature subset sizes (qubits)
# -------------------------------------------------
qubits = np.array([4, 7, 11, 17])

def best_auc_per_k(df: pd.DataFrame, col_auc: str, k_values) -> np.ndarray:
    vals = []
    for k in k_values:
        sub = df[df["n_features"] == k]
        if sub.empty:
            vals.append(np.nan)
        else:
            vals.append(pd.to_numeric(sub[col_auc], errors="coerce").max())
    return np.asarray(vals, dtype=float)

auc_classical = best_auc_per_k(df_classical, "Recalc_Test_ROC_AUC", qubits)
auc_q_scaled  = best_auc_per_k(df_q_scaled,  "QSVC_AUC", qubits)
auc_q_noscale = best_auc_per_k(df_q_noscale, "QSVC_AUC", qubits)

# -------------------------------------------------
# 4. PCHIP smoothing (same as your accuracy figure)
# -------------------------------------------------
def pchip_smooth(x, y, num=200):
    mask = np.isfinite(y)
    x_valid, y_valid = x[mask], y[mask]
    if x_valid.size < 2:
        return x_valid, y_valid
    f = PchipInterpolator(x_valid, y_valid)
    xs = np.linspace(x_valid.min(), x_valid.max(), num)
    return xs, f(xs)

q_cl_s, auc_cl_s = pchip_smooth(qubits, auc_classical)
q_qs_s, auc_qs_s = pchip_smooth(qubits, auc_q_scaled)
q_qn_s, auc_qn_s = pchip_smooth(qubits, auc_q_noscale)

# -------------------------------------------------
# 5. Plot AUC vs qubits
# -------------------------------------------------
plt.figure(figsize=(8, 6))

# Classical SVM
l1, = plt.plot(q_cl_s, auc_cl_s, "-", label="Classical SVM (AUC)")
plt.fill_between(q_cl_s, auc_cl_s, color=l1.get_color(), alpha=0.1)
plt.scatter(qubits, auc_classical, marker="o", color=l1.get_color(),
            edgecolor="black", s=64, zorder=5)

# Quantum + scaling (QSVC)
l2, = plt.plot(q_qs_s, auc_qs_s, "--", label="Quantum + Scaling (QSVC AUC)")
plt.fill_between(q_qs_s, auc_qs_s, color=l2.get_color(), alpha=0.1)
plt.scatter(qubits, auc_q_scaled, marker="s", color=l2.get_color(),
            edgecolor="black", s=64, zorder=5)

# Quantum no scaling (QSVC)
l3, = plt.plot(q_qn_s, auc_qn_s, ":", label="Quantum no Scaling (QSVC AUC)")
plt.fill_between(q_qn_s, auc_qn_s, color=l3.get_color(), alpha=0.1)
plt.scatter(qubits, auc_q_noscale, marker="^", color=l3.get_color(),
            edgecolor="black", s=64, zorder=5)

# -------------------------------------------------
# 6. Dynamic annotations (plateau & peak) in AUC
# -------------------------------------------------
all_vals = np.concatenate([auc_classical, auc_q_scaled, auc_q_noscale])
all_vals = all_vals[np.isfinite(all_vals)]
ymin = all_vals.min() - 0.2
ymax = all_vals.max() + 0.2
plt.ylim(ymin, ymax)

y_offset = 0.003 * (ymax - ymin)

# Plateau at k=7 for quantum + scaling
idx7 = qubits.tolist().index(7)
plateau_val = auc_q_scaled[idx7]
plt.annotate(
    f"Plateau: ~{plateau_val:.2f}",
    xy=(7, plateau_val),
    xytext=(7, plateau_val + y_offset),
    textcoords="data",
    ha="center",
    va="bottom",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8),
    color="black",
)

# Peak at k=11 for quantum no scaling
idx11 = qubits.tolist().index(11)
peak_val = auc_q_noscale[idx11]
plt.annotate(
    f"Peak: ~{peak_val:.2f}",
    xy=(11, peak_val),
    xytext=(11, peak_val + y_offset),
    textcoords="data",
    ha="center",
    va="bottom",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8),
    color="black",
)

# -------------------------------------------------
# 7. Axes / legend / grid
# -------------------------------------------------
plt.xticks(qubits)
plt.xlabel("Number of Qubits (n_features)")
plt.ylabel("Test ROC–AUC (%)")
plt.title("Test ROC–AUC vs. Qubit Count: Classical vs Quantum (Scaled vs Unscaled)")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# =========================
# QSVC Heatmaps Grid — WIDE paper version (improved layout)
# - Wider-than-tall figure (matches your first example style)
# - Subfigs closer together
# - X tick labels straight (no rotation)
# - All labels white; ONLY winners (ties included) in black
# - Black box around every winner
# - Fixed shared color scale with custom ticks
# - Small in-figure caption note
#
# ✅ ADDED (color control function):
#    - You can keep AUTO range (default, same behavior as before), or
#    - Switch to a MANUAL range (vmin/vmax) to make the colors more relaxing.
#
# WHERE TO CHANGE COLORS (ONLY THESE):
#   1) CMAP_NAME  -> colormap name (e.g., "viridis", "YlGn", "GnBu", ...)
#   2) GAMMA      -> contrast (PowerNorm). Lower = smoother, higher = stronger highlights
#   3) USE_MANUAL_RANGE / VMIN / VMAX -> clamp color scale (reduces harsh extremes)
# =========================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.colors import PowerNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Rectangle
from matplotlib.ticker import FormatStrFormatter

# -------------------------
# Config
# -------------------------
CSV_PATH  = "quantum_each_results.csv"
K_LIST    = [4, 7, 11, 17]
ROW_ORDER = ["linear", "full", "circular"]                     # Entanglement (y)
COL_ORDER = ["PauliFeatureMap", "ZFeatureMap", "ZZFeatureMap"] # Feature map (x)

# Short x labels (straight, clean)
COL_LABELS = ["Pauli", "Z", "ZZ"]

# Colormap + gamma (viridis; gamma keeps highs highlighted)
# ✅ CHANGE COLOR HERE:
CMAP_NAME = "viridis"   # <-- change this (e.g., "YlGn", "GnBu", "Greens", ...)
GAMMA     = 4           # <-- change this (lower = more relaxing)

# ✅ OPTIONAL: Clamp the color range (more relaxing)
# Keep defaults so NOTHING changes unless you intentionally set USE_MANUAL_RANGE=True
USE_MANUAL_RANGE = True   # <-- set True to force VMIN/VMAX
VMIN = 70                # <-- only used if USE_MANUAL_RANGE=True
VMAX = 98                # <-- only used if USE_MANUAL_RANGE=True

# Candidate colorbar ticks (filtered to [vmin, vmax])
CBAR_TICKS_CANDIDATES = [65, 80, 90, 95, 98]

CAPTION_NOTE = "Cells = best test accuracy over R=1–4, scaled inputs"

# Typography for paper figure
TITLE_FONTSIZE      = 11
AXIS_LABEL_FONTSIZE = 10
TICK_FONTSIZE       = 9
ANNOT_FONTSIZE      = 9
PANEL_TAG_FONTSIZE  = 10
CAPTION_FONTSIZE    = 9

# Tie tolerance for max detection
MAX_RTL = 1e-12
MAX_ATL = 1e-6

# ---- Layout (WIDE like your first example) ----
FIGSIZE = (10.4, 5.2)  # ✅ wider than tall
DPI     = 320

# tighter spacing + reserve room for colorbar
SUBPLOT_ADJ = dict(
    left=0.07,
    right=0.88,   # ✅ leaves space for the vertical colorbar
    top=0.92,
    bottom=0.16,
    wspace=0.3,
    hspace=0.5
)

CBAR_FRACTION = 0.040
CBAR_PAD      = 0.020

# -------------------------
# Data helpers
# -------------------------
def load_qsvc_matrices(csv_path: str) -> dict:
    """Build per-k 3x3 matrices (rows=entanglement, cols=feature map)
    with max QSVC test accuracy (in %), max over Reps."""
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Could not find {csv_path} in {os.getcwd()}")

    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    needed = {"n_features", "Feature_Map", "Entanglement", "Reps", "QSVC_Test_Acc"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")

    df["n_features"]    = pd.to_numeric(df["n_features"], errors="coerce")
    df["QSVC_Test_Acc"] = pd.to_numeric(df["QSVC_Test_Acc"], errors="coerce")

    s = df["QSVC_Test_Acc"].dropna().astype(float)
    if s.empty:
        raise ValueError("QSVC_Test_Acc column has no valid values.")

    is_fraction = (s.le(1.0 + 1e-12).mean() >= 0.95)
    df["QSVC_Test_Acc_pct"] = df["QSVC_Test_Acc"] * (100.0 if is_fraction else 1.0)

    mats = {}
    for k in K_LIST:
        sub = df[df["n_features"] == k]
        if sub.empty:
            mats[k] = pd.DataFrame(np.nan, index=ROW_ORDER, columns=COL_ORDER)
            continue

        agg = (
            sub.groupby(["Feature_Map", "Entanglement"], dropna=False)["QSVC_Test_Acc_pct"]
               .max()
               .reset_index()
        )

        mat = (
            agg.pivot(index="Entanglement", columns="Feature_Map", values="QSVC_Test_Acc_pct")
               .reindex(index=ROW_ORDER, columns=COL_ORDER)
        )
        mats[k] = mat

    return mats

def compute_shared_scale(mats: dict) -> tuple[float, float]:
    """Global vmin, vmax across all matrices for shared color scale."""
    all_vals = np.concatenate([np.ravel(m.values.astype(float)) for m in mats.values()])
    finite = all_vals[~np.isnan(all_vals)]
    if finite.size == 0:
        return 0.0, 100.0
    return float(np.nanmin(finite)), float(np.nanmax(finite))

# -------------------------
# ✅ ADDED: color range selector (AUTO vs MANUAL)
# -------------------------
def get_color_range(mats: dict) -> tuple[float, float]:
    """
    Decide the color scale range.
    - Default: AUTO range from data (same behavior as your previous code)
    - Optional: MANUAL range using VMIN/VMAX (set USE_MANUAL_RANGE=True)
    """
    if USE_MANUAL_RANGE:
        return float(VMIN), float(VMAX)
    return compute_shared_scale(mats)

# -------------------------
# Plot helpers
# -------------------------
def annotate_cell(ax, value, i, j, is_winner=False):
    """Two-decimal label. Non-winners white; winners black with light halo."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        txt, color = "–", "white"
        stroke = pe.withStroke(linewidth=1.8, foreground="black")
        fw = "normal"
    else:
        txt = f"{value:.2f}"
        if is_winner:
            color = "black"
            stroke = pe.withStroke(linewidth=1.6, foreground="white")
            fw = "bold"
        else:
            color = "white"
            stroke = pe.withStroke(linewidth=2.0, foreground="black")
            fw = "normal"

    ax.text(
        j, i, txt,
        ha="center", va="center",
        fontsize=ANNOT_FONTSIZE,
        color=color,
        fontweight=fw,
        path_effects=[stroke],
        zorder=3
    )

def draw_winner_box(ax, i, j):
    """Black rectangle around a winning cell."""
    rect = Rectangle(
        (j - 0.5, i - 0.5), 1.0, 1.0,
        fill=False, edgecolor="black", linewidth=2.2, zorder=4
    )
    ax.add_patch(rect)

# -------------------------
# Plot
# -------------------------
def show_heatmap_grid(mats: dict, cmap: str = CMAP_NAME, gamma: float = GAMMA) -> None:
    # ✅ uses the new function (AUTO by default, same as before)
    vmin, vmax = get_color_range(mats)
    norm = PowerNorm(gamma=gamma, vmin=vmin, vmax=vmax)

    fig, axes = plt.subplots(
        2, 2,
        figsize=FIGSIZE,
        dpi=DPI,
        constrained_layout=False
    )
    fig.subplots_adjust(**SUBPLOT_ADJ)

    axes = axes.ravel()
    panel_tags = ["(a)", "(b)", "(c)", "(d)"]

    for ax, k, tag in zip(axes, K_LIST, panel_tags):
        mat = mats[k].reindex(index=ROW_ORDER, columns=COL_ORDER)
        vals = mat.values.astype(float)

        # ✅ aspect="auto" so panels fill width (wide look)
        ax.imshow(vals, cmap=cmap, norm=norm, origin="upper", aspect="auto")

        # ticks & labels (straight, not inclined)
        ax.set_xticks(np.arange(len(COL_ORDER)))
        ax.set_xticklabels(COL_LABELS, rotation=0, ha="center", fontsize=TICK_FONTSIZE)
        ax.set_yticks(np.arange(len(ROW_ORDER)))
        ax.set_yticklabels(ROW_ORDER, fontsize=TICK_FONTSIZE)

        ax.set_xlabel("Feature map", fontsize=AXIS_LABEL_FONTSIZE)
        ax.set_ylabel("Entanglement", fontsize=AXIS_LABEL_FONTSIZE)

        # cleaner cell borders
        ax.set_xticks(np.arange(-.5, len(COL_ORDER), 1), minor=True)
        ax.set_yticks(np.arange(-.5, len(ROW_ORDER), 1), minor=True)
        ax.grid(which="minor", color="white", linestyle="-", linewidth=0.9)
        ax.tick_params(which="minor", bottom=False, left=False)

        # winners (ties)
        winners = []
        if np.isfinite(vals).any():
            max_val = np.nanmax(vals)
            winners = np.argwhere(np.isclose(vals, max_val, rtol=MAX_RTL, atol=MAX_ATL))
            winners = [(int(i), int(j)) for i, j in winners]

        # annotations + winner boxes
        for i in range(len(ROW_ORDER)):
            for j in range(len(COL_ORDER)):
                annotate_cell(ax, mat.iat[i, j], i, j, is_winner=((i, j) in winners))
        for (wi, wj) in winners:
            draw_winner_box(ax, wi, wj)

        # title + panel tag
        ax.set_title(f"k = {k} features (mapped to qubits)", fontsize=TITLE_FONTSIZE)
        ax.text(
            0.03, 0.97, tag, transform=ax.transAxes, ha="left", va="top",
            fontsize=PANEL_TAG_FONTSIZE, fontweight="bold",
            bbox=dict(facecolor="white", alpha=0.70, edgecolor="none", boxstyle="round,pad=0.18")
        )

    # shared colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(
        sm,
        ax=axes.tolist(),
        orientation="vertical",
        fraction=CBAR_FRACTION,
        pad=CBAR_PAD
    )
    cbar.set_label("Max QSVC Test Accuracy (%)", fontsize=AXIS_LABEL_FONTSIZE)

    ticks = [t for t in CBAR_TICKS_CANDIDATES if vmin - 1e-9 <= t <= vmax + 1e-9]
    if ticks:
        cbar.set_ticks(ticks)
    cbar.ax.tick_params(labelsize=TICK_FONTSIZE)
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

    # caption note (inside figure, paper-friendly)
    fig.text(0.50, 0.06, CAPTION_NOTE, ha="center", va="center", fontsize=CAPTION_FONTSIZE)

    plt.show()

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    mats = load_qsvc_matrices(CSV_PATH)
    show_heatmap_grid(mats, cmap=CMAP_NAME, gamma=GAMMA)


In [ ]:
# ===============================================
# Reps Envelopes — QSVC + VQC COMBINED (one figure)
# Single super-title; per-panel titles only "k = X"
# ===============================================

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Config (edit these only)
# -------------------------
CSV_PATH   = "quantum_each_results.csv"   # input CSV
K_LIST     = [4, 7, 11, 17]               # panels (k values)
R_TICKS    = [1, 2, 3, 4]                 # Reps shown on x-axis

# --- Colors / styling (your palette) ---
COL_MED     = "#000000"  # black   (median line)
COL_MAX     = "#2ca02c"  # green   (max per R, ▲)
COL_MIN     = "#d62728"  # red     (min per R, ▽)
ENV_COLOR   = "#666666"  # min–max envelope (grey)
ENV_ALPHA   = 0.30
SHADE_COLOR = "#FFD700"  # R≤2 shading (gold)
SHADE_ALPHA = 0.208
GUIDE_COLOR = "#d4b106"
GUIDE_ALPHA = 0.35

# --- Line widths / marker sizes ---
LW_MED  = 2.6
LW_EXT  = 1.8
MS_MED  = 6
MS_EXT  = 6

# --- Typography ---
SUPTITLE = "QSVC & VQC pooled across all configs (feature map × entanglement × estimator × scaling)"
TITLE_FS = 13
AX_FS    = 11
TICK_FS  = 10
LEG_FS   = 10
LABEL_FS = 9

# --- Grid & layout ---
GRID_ALPHA = 0.25

# Figure size / dpi
FIG_SIZE   = (12, 6.0)
FIG_DPI    = 320

# Layout spacing
SUBPLOT_ADJ = dict(top=0.8, bottom=0.24, wspace=0.25, hspace=0.34)

# === Label-only tunables =========================
LABEL_EVERY_CHANGE_ONLY = True
MAX_LABEL_DX_R = {1: +6, 2: +6, 3: +6, 4: -6}
MIN_LABEL_DX_R = {1: +6, 2: +6, 3: +6, 4: -6}
LABEL_BOX_ALPHA = 0.55
BEST_DY_BELOW   = -10
LOSER_DY_OUT    = -7
LOSER_DY_IN     = +13
LOSER_OVERLAP_DELTA_PP = 2.5

# ✅ ONLY for bottom-right subplot (k=17): fix hidden RED (min) labels at R=2 and R=4
K17_MIN_LABEL_OVERRIDES = {
    2: dict(dx=+6, dy=+14, va="bottom", clip_on=False),
    4: dict(dx=-6, dy=-10, va="bottom", clip_on=False),
}

# ✅ NEW: ONLY for subplot (k=11), ONLY R=1: move that ONE RED(min) label
# Change dx/dy below to place it exactly where you want.
K11_MIN_LABEL_OVERRIDES = {
    1: dict(dx=+8, dy=+14, va="bottom", clip_on=False),
}
# =================================================

def _short_scaling(s):
    if pd.isna(s): return "Unknown"
    ss = str(s).strip().lower()
    if re.search(r'\b(z[-\s]?score|zscore)\b', ss): return "Std"
    if re.search(r'\bstandard(ize|ized|ization|isation)?\b', ss): return "Std"
    if re.search(r'\bmin[-\s]?max\b', ss): return "Norm"
    if re.search(r'\bnormaliz(e|ed|ation|ations)?\b', ss): return "Norm"
    if re.search(r'\brescal(e|ed|ing)?\b', ss): return "Norm"
    return str(s)

def short_feature_map(s: str) -> str:
    s = str(s)
    if "ZZFeatureMap" in s: return "ZZ"
    if "ZFeatureMap"  in s: return "Z"
    if "Pauli"        in s: return "Pauli"
    return s

def short_ent(s: str) -> str:
    return {"linear":"linear", "full":"full", "circular":"circular"}.get(str(s), str(s))

def build_cfg_label(row: pd.Series) -> str:
    parts = [f"{short_feature_map(row.get('Feature_Map',''))}-{short_ent(row.get('Entanglement',''))}"]
    if "Estimator" in row and pd.notna(row["Estimator"]):
        est = str(row["Estimator"])
        parts.append("LR" if "ogistic" in est else ("RF" if "Random Forest" in est else est))
    sc_val = row.get("Scaling_short")
    if pd.isna(sc_val) and "Scaling" in row and pd.notna(row["Scaling"]):
        sc_val = _short_scaling(row["Scaling"])
    if sc_val:
        parts.append(str(sc_val))
    head = parts[0]
    tail = ":".join([p for p in parts[1:] if p])
    return f"{head}:{tail}" if tail else head

def load_combined(csv_path: str, verbose: bool = False) -> pd.DataFrame:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    needed = {"n_features", "Feature_Map", "Entanglement", "Reps"}
    if not needed.issubset(df.columns):
        raise ValueError(f"Missing columns: {needed - set(df.columns)}")

    df["n_features"] = pd.to_numeric(df["n_features"], errors="coerce")
    df["Reps"]       = pd.to_numeric(df["Reps"], errors="coerce")

    metric_cols = [c for c in ["QSVC_Test_Acc", "VQC_Test_Acc"] if c in df.columns]
    if not metric_cols:
        raise ValueError("Expected at least one of 'QSVC_Test_Acc' or 'VQC_Test_Acc'.")
    for c in metric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    long = df.melt(
        id_vars=[c for c in df.columns if c not in metric_cols],
        value_vars=metric_cols,
        var_name="AlgMetric",
        value_name="Test_Acc"
    )
    long["Algorithm"] = long["AlgMetric"].replace({"QSVC_Test_Acc": "QSVC", "VQC_Test_Acc": "VQC"})
    long = long.drop(columns=["AlgMetric"])

    s = long["Test_Acc"].dropna().astype(float)
    is_fraction = (not s.empty) and (s.le(1.0 + 1e-12).mean() >= 0.95)
    long["acc_pct"] = long["Test_Acc"] * (100.0 if is_fraction else 1.0)

    if "Scaling" in long.columns:
        long["Scaling_short"] = long["Scaling"].map(_short_scaling)
    else:
        long["Scaling_short"] = "Unknown"

    keep = ["Algorithm","n_features","Reps","Feature_Map","Entanglement","acc_pct"]
    for extra in ["Estimator","Scaling","Scaling_short"]:
        if extra in long.columns: keep.append(extra)
    long = long[keep].dropna(subset=["n_features","Reps","acc_pct"]).copy()

    cfg_keys = ["Algorithm","n_features","Reps","Feature_Map","Entanglement"]
    if "Estimator" in long.columns:     cfg_keys.append("Estimator")
    if "Scaling"  in long.columns:      cfg_keys.append("Scaling")
    if "Scaling_short" in long.columns: cfg_keys.append("Scaling_short")

    base = (long.groupby(cfg_keys, dropna=False)["acc_pct"]
                 .mean().reset_index().rename(columns={"acc_pct":"acc"}))
    base["cfg_label"] = base.apply(build_cfg_label, axis=1)
    return base

def summarize_all(base: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k in K_LIST:
        sub_k = base[base["n_features"] == k]
        if sub_k.empty: continue
        for r in R_TICKS:
            slab = sub_k[sub_k["Reps"] == r]
            if slab.empty: continue
            vals = slab["acc"].to_numpy(float)
            vmin = float(np.nanmin(vals))
            vmax = float(np.nanmax(vals))
            vmed = float(np.nanmedian(vals))
            lab_max = slab.loc[np.isclose(slab["acc"], vmax, atol=1e-12), "cfg_label"].sort_values().iloc[0]
            lab_min = slab.loc[np.isclose(slab["acc"], vmin, atol=1e-12), "cfg_label"].sort_values().iloc[0]
            rows.append(dict(n_features=k, R=r,
                             median_acc=vmed, min_acc=vmin, max_acc=vmax,
                             label_min=lab_min, label_max=lab_max))
    return pd.DataFrame(rows).sort_values(["n_features","R"])

def compute_yaxis(df: pd.DataFrame):
    gmin = float(df["min_acc"].min()) if not df["min_acc"].isna().all() else 50.0
    y_min = max(0.0, np.floor((gmin - 0.01)/5)*5 - 2)
    y_max = 100.5
    start_tick = int(np.ceil(max(0, y_min)/10.0)*10)
    yticks = list(range(start_tick, 101, 10))
    if y_min < start_tick:
        yticks = [int(np.floor(y_min/5.0)*5)] + yticks
    return y_min, y_max, yticks

def plot_combined_grid(tidy: pd.DataFrame):
    assert {"n_features","R","median_acc","min_acc","max_acc"}.issubset(tidy.columns)
    y_min, y_max, yticks = compute_yaxis(tidy)

    fig, axes = plt.subplots(2, 2, figsize=FIG_SIZE, dpi=FIG_DPI, constrained_layout=False)
    fig.subplots_adjust(**SUBPLOT_ADJ)
    fig.suptitle(SUPTITLE, fontsize=TITLE_FS+1, y=0.9)

    axes = axes.ravel()
    for ax, k in zip(axes, K_LIST):
        sk = tidy[tidy["n_features"] == k].sort_values("R")
        if sk.empty:
            ax.set_visible(False); continue

        R      = sk["R"].to_numpy()
        y_med  = sk["median_acc"].to_numpy()
        y_minK = sk["min_acc"].to_numpy()
        y_maxK = sk["max_acc"].to_numpy()

        ax.axvspan(0.9, 2.1, color=SHADE_COLOR, alpha=SHADE_ALPHA, zorder=0)
        ax.axvline(1.008, color=GUIDE_COLOR, alpha=GUIDE_ALPHA, lw=1.2, zorder=0)

        ax.fill_between(R, y_minK, y_maxK, color=ENV_COLOR, alpha=ENV_ALPHA,
                        linewidth=0, zorder=1, label="Min–Max envelope")

        ax.plot(R, y_med,  "-o", color=COL_MED, lw=LW_MED, ms=MS_MED, zorder=5, label="Median across configs")
        ax.plot(R, y_maxK, "-^", color=COL_MAX, lw=LW_EXT, ms=MS_EXT, zorder=6, label="Max per R")
        ax.plot(R, y_minK, "-v", color=COL_MIN, lw=LW_EXT, ms=MS_EXT,
                markerfacecolor="white", markeredgecolor=COL_MIN, markeredgewidth=1.2,
                zorder=6, label="Min per R")

        # Green labels (max): always outside/below
        prev_max = None
        for r, y, lab in sk[["R","max_acc","label_max"]].itertuples(index=False):
            if isinstance(lab, str) and lab.strip() and (not LABEL_EVERY_CHANGE_ONLY or r == 1 or lab != prev_max):
                dx = MAX_LABEL_DX_R.get(int(r), +6)
                ax.annotate(
                    lab, xy=(r, y), xytext=(dx, -10), textcoords="offset points",
                    ha="left" if dx >= 0 else "right", va="top",
                    fontsize=LABEL_FS, color=COL_MAX,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=LABEL_BOX_ALPHA, boxstyle="round,pad=0.15"),
                    arrowprops=dict(arrowstyle="-", color=COL_MAX, lw=0.8),
                    clip_on=True
                )
                prev_max = lab

        # Red labels (min): default below; special fixes via overrides
        prev_min_y = None
        for r, y, lab in sk[["R","min_acc","label_min"]].itertuples(index=False):
            if not (isinstance(lab, str) and lab.strip()):
                continue

            r_int = int(r)
            dx = MIN_LABEL_DX_R.get(r_int, +6)
            dy, va = (LOSER_DY_OUT, "top")
            clip_on = True

            if k == 4 and r_int == 3:
                dy, va = (LOSER_DY_IN, "bottom")
            elif prev_min_y is not None and abs(y - prev_min_y) <= LOSER_OVERLAP_DELTA_PP:
                dy, va = (LOSER_DY_IN, "bottom")

            # ✅ k=17 overrides (ONLY R=2 and R=4)
            if k == 17 and r_int in K17_MIN_LABEL_OVERRIDES:
                ov = K17_MIN_LABEL_OVERRIDES[r_int]
                dx = ov.get("dx", dx)
                dy = ov.get("dy", dy)
                va = ov.get("va", va)
                clip_on = ov.get("clip_on", clip_on)

            # ✅ NEW: k=11 override (ONLY R=1)
            if k == 11 and r_int in K11_MIN_LABEL_OVERRIDES:
                ov = K11_MIN_LABEL_OVERRIDES[r_int]
                dx = ov.get("dx", dx)
                dy = ov.get("dy", dy)
                va = ov.get("va", va)
                clip_on = ov.get("clip_on", clip_on)

            ax.annotate(
                lab, xy=(r, y), xytext=(dx, dy), textcoords="offset points",
                ha="left" if dx >= 0 else "right", va=va,
                fontsize=LABEL_FS, color=COL_MIN,
                bbox=dict(facecolor="white", edgecolor="none", alpha=LABEL_BOX_ALPHA, boxstyle="round,pad=0.15"),
                arrowprops=dict(arrowstyle="-", color=COL_MIN, lw=0.8),
                clip_on=clip_on
            )
            prev_min_y = y

        # Axes cosmetics
        ax.set_title(f"k = {k}", fontsize=TITLE_FS)
        ax.set_xlabel("Reps", fontsize=AX_FS)
        ax.xaxis.set_label_coords(0.5, -0.06)
        ax.set_ylabel("Test Accuracy (%)", fontsize=AX_FS)
        ax.set_xticks(R_TICKS); ax.set_xlim(1, 4)
        ax.set_yticks(yticks);  ax.set_ylim(y_min, y_max)
        ax.tick_params(labelsize=TICK_FS)
        ax.grid(True, axis="both", alpha=GRID_ALPHA)

    # Shared legend (no caption)
    handles = [
        plt.Line2D([0], [0], color=COL_MED, lw=LW_MED,  ls="-", marker="o", label="Median across configs"),
        plt.Line2D([0], [0], color=COL_MAX, lw=LW_EXT,  marker="^", label="Max per R"),
        plt.Line2D([0], [0], color=COL_MIN, lw=LW_EXT,  marker="v",
                   markerfacecolor="white", markeredgecolor=COL_MIN, markeredgewidth=1.2,
                   label="Min per R"),
        plt.Line2D([0], [0], color=ENV_COLOR, lw=6, alpha=ENV_ALPHA, label="Min–Max envelope"),
    ]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.12),
               ncol=4, frameon=False, fontsize=LEG_FS)

    plt.show()
    return fig

if __name__ == "__main__":
    base_all = load_combined(CSV_PATH, verbose=False)
    tidy     = summarize_all(base_all)
    plot_combined_grid(tidy)


In [ ]:
# ======= QSVC vs VQC — one-cell dashboard (best per k; unified floor 50; headroom capped at 105; safe peak labels) =======
import numpy as np                      # numerical arrays & math
import pandas as pd                     # tabular data handling
import matplotlib.pyplot as plt         # plotting API
from matplotlib.patches import Patch    # legend patch (for IQR key)
from matplotlib.lines import Line2D     # legend line entries (QSVC/VQC)
from matplotlib.ticker import MultipleLocator  # (kept available if you later want minor ticks)

WIDE_CSV = "quantum_each_results.csv"   # expected input file in “wide” format (your experiment dump)
FORCE_YLIM = (50, 105)                  # default y-range floor/ceiling; we later cap panels to max 105

# ---------- helpers (UNCHANGED data handling) ----------
def _percentize(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")                     # ensure numeric dtype, NaNs where needed
    if s.notna().sum() == 0:                                  # if everything is NA, return as-is
        return s
    mn, mx = float(s.min(skipna=True)), float(s.max(skipna=True))  # detect scale (0–1 vs 0–100)
    return s*100.0 if (mx <= 1.05 and mn >= 0.0) else s       # convert to % only if data is in 0–1

def _wide_to_tidy(df_wide: pd.DataFrame) -> pd.DataFrame:
    df = df_wide.copy()                                       # work on a copy
    df.columns = df.columns.str.strip()                       # normalize column names
    if "k" not in df.columns and "n_features" in df.columns:  # support alternative column name for k
        df["k"] = pd.to_numeric(df["n_features"], errors="coerce")
    else:
        df["k"] = pd.to_numeric(df.get("k", np.nan), errors="coerce")
    for c in ["Feature_Map","Feature_M","Feature_N","Feature"]:  # unify feature-map column name
        if c in df.columns:
            df = df.rename(columns={c:"Feature_Map"})
            break

    common = ["k","Scaling","Estimator","Feature_Map","Entanglement","Reps","Duration_sec"]  # meta columns to keep

    # Map QSVC columns from wide → tidy
    qsvc_map = {
        "Model": "QSVC",
        "Test_Acc_%": df.get("QSVC_Test_Acc", np.nan),
        "AUC_%":      df.get("QSVC_AUC", np.nan),
        "F1_%":       df.get("QSVC_F1",  np.nan),
        "TP":         df.get("QSVC_TP",  np.nan),
        "FP":         df.get("QSVC_FP",  np.nan),
        "TN":         df.get("QSVC_TN",  np.nan),
        "FN":         df.get("QSVC_FN",  np.nan),
    }
    # Map VQC columns from wide → tidy
    vqc_map = {
        "Model": "VQC",
        "Test_Acc_%": df.get("VQC_Test_Acc", np.nan),
        "AUC_%":      df.get("VQC_AUC", np.nan),
        "F1_%":       df.get("VQC_F1",  np.nan),
        "TP":         df.get("VQC_TP",  np.nan),
        "FP":         df.get("VQC_FP",  np.nan),
        "TN":         df.get("VQC_TN",  np.nan),
        "FN":         df.get("VQC_FN",  np.nan),
    }

    def build_one(model_map):
        d = df[common].copy()                                  # start from meta columns
        for k, v in model_map.items():                         # add model-specific columns
            d[k] = v
        for c in ["Test_Acc_%","AUC_%","F1_%"]:                # normalize to percentages if needed
            d[c] = _percentize(d[c])
        for i, r in d.iterrows():                              # derive Sens/Spec/Prec from confusion counts
            tp, fp, tn, fn = (r.get("TP"), r.get("FP"), r.get("TN"), r.get("FN"))
            if pd.notna(tp) and pd.notna(fp) and pd.notna(tn) and pd.notna(fn):
                tp, fp, tn, fn = map(float, (tp, fp, tn, fn))
                sens = 100.0*(tp/(tp+fn)) if (tp+fn)>0 else np.nan
                spec = 100.0*(tn/(tn+fp)) if (tn+fp)>0 else np.nan
                prec = 100.0*(tp/(tp+fp)) if (tp+fp)>0 else np.nan
                d.loc[i, "Sensitivity_%"] = sens
                d.loc[i, "Specificity_%"]  = spec
                d.loc[i, "Precision_%"]    = prec
                # If F1 was missing, compute from precision & sensitivity
                if (pd.isna(r.get("F1_%")) or not np.isfinite(r.get("F1_%"))) and np.isfinite(prec) and np.isfinite(sens) and (prec+sens)>0:
                    d.loc[i, "F1_%"] = 200.0*(prec*sens)/(prec+sens)/100.0
        keep = ["Model","k","Scaling","Estimator","Feature_Map","Entanglement","Reps",
                "Test_Acc_%","AUC_%","F1_%","Sensitivity_%","Specificity_%","Precision_%",
                "TP","FP","TN","FN","Duration_sec"]            # final tidy schema
        return d[keep]

    tidy = pd.concat([build_one(qsvc_map), build_one(vqc_map)], ignore_index=True)  # QSVC and VQC stacked
    tidy["k"] = pd.to_numeric(tidy["k"], errors="coerce").astype(int)               # ensure integer k
    return tidy.reset_index(drop=True)                                              # clean index

def _best_per_k_model(tidy: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (k, model), g in tidy.groupby(["k","Model"], dropna=True):                  # per k & model
        g = g.copy()
        for c in ["F1_%","AUC_%","Test_Acc_%"]:
            if c not in g.columns: g[c] = np.nan                                    # safety
        # Sort by F1, then AUC, then Accuracy (all descending); mergesort keeps stability (tie-break)
        g = g.sort_values(["F1_%","AUC_%","Test_Acc_%"], ascending=[False, False, False], kind="mergesort")
        rows.append(g.iloc[0])                                                      # pick the best row
    return pd.DataFrame(rows).sort_values(["k","Model"]).reset_index(drop=True)     # return best-per-(k,model)

# ---- stats helpers ----
def _wilson_ci(successes, total, z: float = 1.96):
    # 95% Wilson score interval for binomial proportion, returned as percentages
    if total is None or total <= 0 or not np.isfinite(total) or not np.isfinite(successes):
        return (np.nan, np.nan)
    k = float(successes); n = float(total); p = k/n
    denom  = 1.0 + (z**2)/n
    center = (p + (z**2)/(2*n)) / denom
    half   = (z/denom) * np.sqrt((p*(1-p)/n) + (z**2)/(4*n**2))
    return 100.0*(center - half), 100.0*(center + half)

def _iqr_by_k_model(tidy_df: pd.DataFrame, metric: str):
    # Compute q25/median/q75 per (k, model) for shading IQR bands
    df = tidy_df.copy()
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    q = (df.groupby(["k","Model"])[metric]
           .quantile([0.25, 0.5, 0.75])
           .unstack(-1)
           .rename(columns={0.25:"q25", 0.5:"q50", 0.75:"q75"})
           .reset_index())
    return q

# ---------- build inputs ----------
_wide = pd.read_csv(WIDE_CSV)                          # load the wide-format CSV
tidy  = _wide_to_tidy(_wide)                           # convert to tidy (one row per config)
best  = _best_per_k_model(tidy)                        # select best row per (k, model)
iqr_auc = _iqr_by_k_model(tidy, "AUC_%")               # IQR table for AUC (used for bands)
iqr_f1  = _iqr_by_k_model(tidy, "F1_%")                # IQR table for F1  (used for bands)

# precompute accuracy CIs for best rows
def _add_accuracy_ci(best_df):
    lo, hi = [], []
    for _, r in best_df.iterrows():
        tp, tn, fp, fn = r.get("TP"), r.get("TN"), r.get("FP"), r.get("FN")     # confusion counts
        if all(pd.notna([tp, tn, fp, fn])):
            l, h = _wilson_ci(tp+tn, tp+tn+fp+fn)                               # Wilson CI on accuracy
        else:
            l, h = (np.nan, np.nan)
        lo.append(l); hi.append(h)
    out = best_df.copy()
    out["Acc_lo95"] = lo; out["Acc_hi95"] = hi                                  # attach lower/upper CI
    return out

best = _add_accuracy_ci(best)                                                   # enrich “best” with CI

# ---------- DASHBOARD ----------
def dashboard_qsvc_vqc(best_df: pd.DataFrame,
                       force_ylim=FORCE_YLIM,
                       iqr_auc_df=None, iqr_f1_df=None):
    # Define six metric panels (column key, panel title)
    metrics = [
        ("Test_Acc_%","Accuracy (%)"),
        ("AUC_%","AUC (%)"),
        ("F1_%","F1 (%)"),
        ("Sensitivity_%","Sensitivity (%)"),
        ("Specificity_%","Specificity (%)"),
        ("Precision_%","Precision (%)"),
    ]
    ks = sorted(best_df["k"].astype(int).unique().tolist())  # x-axis values (feature/qubit counts)
    x  = np.array(ks, float)                                  # x as numpy array for plotting

    # same hues, slightly darker (requested look)
    colors  = {"QSVC":"#1d5fa7", "VQC":"#cc6a13"}             # line colors (QSVC blue, VQC orange)
    markers = {"QSVC":"o","VQC":"s"}                          # line markers
    band_alpha = 0.08                                         # transparency for IQR shading

    fig, axes = plt.subplots(2, 3, figsize=(14.2, 7.6), dpi=320, sharex=True)  # 2×3 grid figure
    axes = axes.ravel()                                       # flatten axes list for easy iteration

    panel_labels = ["(a)","(b)","(c)","(d)","(e)","(f)"]      # panel letters for each subplot

    def _clamp(arr, lo, hi):
        arr = np.asarray(arr, float)
        return np.clip(arr, lo, hi)                           # bound an array to [lo, hi]

    def _shade_iqr(ax, iqr_df, model_name, color, ylim_tuple):
        # Draw IQR band (q25–q75) for a given metric/model, clipped to local y-limits
        if iqr_df is None: return
        sub = iqr_df[iqr_df["Model"]==model_name].set_index("k").reindex(ks)
        if {"q25","q75"}.issubset(sub.columns):
            q25 = _clamp(sub["q25"].astype(float).to_numpy(), *ylim_tuple)
            q75 = _clamp(sub["q75"].astype(float).to_numpy(), *ylim_tuple)
            mask = np.isfinite(q25) & np.isfinite(q75)
            if mask.sum() >= 2:
                ax.fill_between(x[mask], q25[mask], q75[mask],
                                color=color, alpha=band_alpha, linewidth=0, zorder=1, clip_on=True)

    for idx, (ax, (col, label)) in enumerate(zip(axes, metrics)):
        # panel letter outside (so it never overlaps curves)
        ax.text(-0.16, 1.02, panel_labels[idx], transform=ax.transAxes,
                ha="left", va="bottom", fontsize=11, weight="bold", clip_on=False)

        qb = (best_df[best_df["Model"]=="QSVC"].set_index("k")[col]
              .reindex(ks).astype(float).to_numpy())          # y-values for QSVC
        vb = (best_df[best_df["Model"]=="VQC"].set_index("k")[col]
              .reindex(ks).astype(float).to_numpy())          # y-values for VQC

        # ---------- strict headroom control (max 105%) ----------
        y_all = np.concatenate([qb, vb])                      # look at both series to decide ticks
        y_top = np.nanmax(y_all) if np.isfinite(y_all).any() else force_ylim[1]
        y_min = force_ylim[0]
        y_max = 105                                           # hard cap at 105, as requested
        local_ylim = (y_min, y_max)                           # per-panel limits

        # IQR bands (AUC/F1 only) clamped to local y-limits
        if label.startswith("AUC"):
            _shade_iqr(ax, iqr_auc, "QSVC", colors["QSVC"], local_ylim)
            _shade_iqr(ax, iqr_auc, "VQC",  colors["VQC"],  local_ylim)
        if label.startswith("F1"):
            _shade_iqr(ax, iqr_f1,  "QSVC", colors["QSVC"], local_ylim)
            _shade_iqr(ax, iqr_f1,  "VQC",  colors["VQC"],  local_ylim)

        # metric lines (QSVC then VQC)
        ax.plot(x, qb, "-", marker=markers["QSVC"], color=colors["QSVC"], lw=1.9, ms=6, zorder=3, label="QSVC")
        ax.plot(x, vb, "-", marker=markers["VQC"],  color=colors["VQC"],  lw=1.9, ms=6, zorder=3, label="VQC")

        # ▲ peak markers + labels (QSVC uses “safe” placement; VQC label is below/right)
        for yvals, colr in [(qb, colors["QSVC"]), (vb, colors["VQC"])]:
            if np.isfinite(yvals).any():
                imax = int(np.nanargmax(yvals))               # index of maximum point
                ypk  = float(yvals[imax])                     # peak value
                ax.scatter(x[imax], ypk, marker="^", s=42,    # triangle marker on the peak
                           facecolor="none", edgecolor=colr, lw=1.3, zorder=4)

                if colr == colors["VQC"]:
                    # VQC: always annotate under & to the right of the peak (your request)
                    ax.annotate(f"{ypk:.1f}%", (x[imax], ypk),
                                textcoords="offset points", xytext=(+10, -10),
                                ha="left", va="top", fontsize=9, color=colr, clip_on=False)
                else:
                    # QSVC: try above; if too close to the top go below; otherwise omit
                    margin = 0.8
                    if ypk <= y_max - margin:
                        ax.annotate(f"{ypk:.1f}%", (x[imax], ypk),
                                    textcoords="offset points", xytext=(0, 6),
                                    ha="center", va="bottom", fontsize=9, color=colr, clip_on=False)
                    elif ypk <= y_max - 0.2:
                        ax.annotate(f"{ypk:.1f}%", (x[imax], ypk),
                                    textcoords="offset points", xytext=(0, -10),
                                    ha="center", va="top", fontsize=9, color=colr, clip_on=False)
                    # else: too tight to the top → leave marker only

        # Accuracy CI whiskers (95% Wilson) clamped to local y-limits
        if label.startswith("Accuracy"):
            for model, y, ecolor in [("QSVC",qb,colors["QSVC"]), ("VQC",vb,colors["VQC"])]:
                sub = best_df[best_df["Model"]==model].set_index("k")
                lo = _clamp(sub["Acc_lo95"].reindex(ks).to_numpy(float), *local_ylim)
                hi = _clamp(sub["Acc_hi95"].reindex(ks).to_numpy(float), *local_ylim)
                y  = _clamp(y, *local_ylim)
                yerr = np.vstack([y-lo, hi-y])
                yerr = np.where(np.isfinite(yerr), yerr, np.nan)
                ax.errorbar(x, y, yerr=yerr, fmt="none",
                            ecolor=ecolor, elinewidth=0.9, capsize=2, alpha=0.5, zorder=2)

        # Sensitivity CI whiskers + dotted reference at 95%
        if label.startswith("Sensitivity"):
            for model, y, ecolor in [("QSVC",qb,colors["QSVC"]), ("VQC",vb,colors["VQC"])]:
                sub = best_df[best_df["Model"]==model].set_index("k")
                lo, hi = [], []
                for kk in ks:
                    tp = sub.at[kk,"TP"] if ("TP" in sub.columns and kk in sub.index) else np.nan
                    fn = sub.at[kk,"FN"] if ("FN" in sub.columns and kk in sub.index) else np.nan
                    l, h = _wilson_ci(tp, tp+fn) if pd.notna(tp) and pd.notna(fn) else (np.nan, np.nan)
                    lo.append(l); hi.append(h)
                y  = _clamp(y, *local_ylim)
                lo = _clamp(np.array(lo), *local_ylim)
                hi = _clamp(np.array(hi), *local_ylim)
                yerr = np.vstack([y-lo, hi-y]); yerr = np.where(np.isfinite(yerr), yerr, np.nan)
                ax.errorbar(x, y, yerr=yerr, fmt="none",
                            ecolor=ecolor, elinewidth=0.9, capsize=2, alpha=0.5, zorder=2)
            ax.axhline(95.0, ls=(0,(2,2)), lw=1.0, color="#9aa5b1", alpha=0.6, zorder=0)

        # Specificity CI whiskers + dotted reference at 95%
        if label.startswith("Specificity"):
            for model, y, ecolor in [("QSVC",qb,colors["QSVC"]), ("VQC",vb,colors["VQC"])]:
                sub = best_df[best_df["Model"]==model].set_index("k")
                lo, hi = [], []
                for kk in ks:
                    tn = sub.at[kk,"TN"] if ("TN" in sub.columns and kk in sub.index) else np.nan
                    fp = sub.at[kk,"FP"] if ("FP" in sub.columns and kk in sub.index) else np.nan
                    l, h = _wilson_ci(tn, tn+fp) if pd.notna(tn) and pd.notna(fp) else (np.nan, np.nan)
                    lo.append(l); hi.append(h)
                y  = _clamp(y, *local_ylim)
                lo = _clamp(np.array(lo), *local_ylim)
                hi = _clamp(np.array(hi), *local_ylim)
                yerr = np.vstack([y-lo, hi-y]); yerr = np.where(np.isfinite(yerr), yerr, np.nan)
                ax.errorbar(x, y, yerr=yerr, fmt="none",
                            ecolor=ecolor, elinewidth=0.9, capsize=2, alpha=0.5, zorder=2)
            ax.axhline(95.0, ls=(0,(2,2)), lw=1.0, color="#9aa5b1", alpha=0.6, zorder=0)

        # ticks/grid bounded by local ylim; top tick never exceeds 105
        ax.set_ylim(*local_ylim)                               # enforce [50, 105] range
        top_tick = min(105, int(np.ceil(y_max / 10.0)) * 10)   # compute top y-tick (≤ 105)
        ax.set_yticks(np.arange(y_min, top_tick + 1, 10))      # 10% step ticks
        ax.grid(alpha=0.15, which="major", zorder=0)           # light major grid
        ax.set_title(label, fontsize=12)                       # panel title
        ax.set_xticks(x)                                       # x ticks at each k

    # compact legend (QSVC, VQC, single IQR sample with hatch)
    custom_legend = [
        Line2D([0],[0], color=colors["QSVC"], marker='o', lw=1.9, ms=6, label="QSVC"),
        Line2D([0],[0], color=colors["VQC"],  marker='s', lw=1.9, ms=6, label="VQC"),
        Patch(facecolor="none", edgecolor="#666666", hatch="///", label="IQR (AUC/F1)"),
    ]

    fig.suptitle("QSVC vs VQC — Best per k (F1 → AUC → Acc tie-break)", fontsize=14, y=0.9) # figure title
    for ax in axes[3:]:
        ax.set_xlabel("k (features/qubits)")                   # x-label only on bottom row
    axes[0].set_ylabel("Metric (%)"); axes[3].set_ylabel("Metric (%)")  # y-labels on left column

    # legend centered below the grid; no frame
    fig.legend(handles=custom_legend, loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.0002))

    # layout: reserve bottom area for legend and leave a bit less space under the title
    fig.tight_layout(rect=[0, 0.06, 1, 0.90])  # [left, bottom, right, top] in figure coords

    plt.show()                    # render the figure
    return fig                    # return the Matplotlib Figure object

# ---- run it ----
dashboard_qsvc_vqc(best, force_ylim=FORCE_YLIM, iqr_auc_df=iqr_auc, iqr_f1_df=iqr_f1)  # execute with prepared inputs


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ----- data (QSVC only) -----
data = [
    {"k": 4,  "FN": 2, "FP": 2},
    {"k": 7,  "FN": 0, "FP": 2},
    {"k": 11, "FN": 1, "FP": 1},
    {"k": 17, "FN": 1, "FP": 2},
]

ks  = [d["k"] for d in data]
FN  = np.array([d["FN"] for d in data], dtype=float)
FP  = np.array([d["FP"] for d in data], dtype=float)
tot = FN + FP

# ----- plotting -----
fig, ax = plt.subplots(figsize=(7.2, 4.2), dpi=300)

width = 0.6
fn_color = "#d62728"   # strong red to emphasize FN
fp_color = "#f4a261"   # lighter orange for FP

# stacked bars
ax.bar(ks, FN, width=width, color=fn_color, zorder=3)
ax.bar(ks, FP, width=width, bottom=FN, color=fp_color, zorder=3)

# annotate each segment
ymax = int(tot.max()) + 1
for x, fn, fp in zip(ks, FN, FP):
    # FN label (inside bottom segment, or just above baseline if 0)
    if fn > 0:
        ax.text(x, fn/2, f"{int(fn)}", ha="center", va="center",
                color="white", fontsize=10, zorder=4)
    else:
        ax.text(x, 0.05, "0", ha="center", va="bottom",
                color=fn_color, fontsize=10, zorder=4)
    # FP label (inside top segment)
    if fp > 0:
        ax.text(x, fn + fp/2, f"{int(fp)}", ha="center", va="center",
                color="black", fontsize=10, zorder=4)
    else:
        ax.text(x, fn + 0.05, "0", ha="center", va="bottom",
                color="black", fontsize=10, zorder=4)

# cosmetics
ax.set_xticks(ks)
ax.set_xlabel("k (features/qubits)", fontsize=11)
ax.set_ylabel("Count (test set)", fontsize=11)
ax.set_title("QSVC error profile by k (scaled inputs)", fontsize=13)
ax.set_ylim(0, ymax if ymax >= 3 else 3)  # ensure a bit of headroom
ax.grid(axis="y", alpha=0.15, zorder=0)

# subtle note under x-axis
fig.subplots_adjust(bottom=0.22)  # make room for the note
fig.text(0.5, 0.06, "In screening, minimize FN", ha="center", fontsize=10, color="#444")

# optional tiny legend (comment out if you don't want it)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=fn_color, label="FN"),
                   Patch(facecolor=fp_color, label="FP")],
          loc="upper right", frameon=False)

plt.show()


In [ ]:
# ============================================
# QSVC FNR / FPR vs k (with 95% CI)
# Wide CSV -> tidy (same as dashboard) -> best QSVC per k.
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional

WIDE_CSV = "quantum_each_results.csv"  # your wide-format dump

# ---------- helpers: shared-style with dashboard ----------

def _percentize(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0:
        return s
    mn, mx = float(s.min(skipna=True)), float(s.max(skipna=True))
    return s * 100.0 if (mx <= 1.05 and mn >= 0.0) else s  # convert 0–1 → 0–100

def _wide_to_tidy(df_wide: pd.DataFrame) -> pd.DataFrame:
    df = df_wide.copy()
    df.columns = df.columns.str.strip()

    # unify "k"
    if "k" not in df.columns and "n_features" in df.columns:
        df["k"] = pd.to_numeric(df["n_features"], errors="coerce")
    else:
        df["k"] = pd.to_numeric(df.get("k", np.nan), errors="coerce")

    # unify Feature_Map
    for c in ["Feature_Map", "Feature_M", "Feature_N", "Feature"]:
        if c in df.columns:
            df = df.rename(columns={c: "Feature_Map"})
            break

    common = [
        "k", "Scaling", "Estimator", "Feature_Map",
        "Entanglement", "Reps", "Duration_sec"
    ]

    # map QSVC wide columns
    qsvc_map = {
        "Model": "QSVC",
        "Test_Acc_%": df.get("QSVC_Test_Acc", np.nan),
        "AUC_%":      df.get("QSVC_AUC",       np.nan),
        "F1_%":       df.get("QSVC_F1",        np.nan),
        "TP":         df.get("QSVC_TP",        np.nan),
        "FP":         df.get("QSVC_FP",        np.nan),
        "TN":         df.get("QSVC_TN",        np.nan),
        "FN":         df.get("QSVC_FN",        np.nan),
    }

    # map VQC wide columns (kept to stay consistent with dashboard)
    vqc_map = {
        "Model": "VQC",
        "Test_Acc_%": df.get("VQC_Test_Acc", np.nan),
        "AUC_%":      df.get("VQC_AUC",       np.nan),
        "F1_%":       df.get("VQC_F1",        np.nan),
        "TP":         df.get("VQC_TP",        np.nan),
        "FP":         df.get("VQC_FP",        np.nan),
        "TN":         df.get("VQC_TN",        np.nan),
        "FN":         df.get("VQC_FN",        np.nan),
    }

    def build_one(model_map):
        d = df[common].copy()
        for k, v in model_map.items():
            d[k] = v

        # normalize metrics
        for c in ["Test_Acc_%", "AUC_%", "F1_%"]:
            d[c] = _percentize(d[c])

        # derive Sensitivity / Specificity / Precision if counts are present
        for i, r in d.iterrows():
            tp, fp, tn, fn = r.get("TP"), r.get("FP"), r.get("TN"), r.get("FN")
            if pd.notna(tp) and pd.notna(fp) and pd.notna(tn) and pd.notna(fn):
                tp, fp, tn, fn = map(float, (tp, fp, tn, fn))
                sens = 100.0 * (tp / (tp + fn)) if (tp + fn) > 0 else np.nan
                spec = 100.0 * (tn / (tn + fp)) if (tn + fp) > 0 else np.nan
                prec = 100.0 * (tp / (tp + fp)) if (tp + fp) > 0 else np.nan
                d.loc[i, "Sensitivity_%"] = sens
                d.loc[i, "Specificity_%"] = spec
                d.loc[i, "Precision_%"]   = prec

                # fill F1 if missing
                if (pd.isna(r.get("F1_%")) or not np.isfinite(r.get("F1_%"))) \
                   and np.isfinite(prec) and np.isfinite(sens) and (prec + sens) > 0:
                    d.loc[i, "F1_%"] = 200.0 * (prec * sens) / (prec + sens) / 100.0

        keep = [
            "Model", "k", "Scaling", "Estimator", "Feature_Map", "Entanglement",
            "Reps", "Test_Acc_%", "AUC_%", "F1_%", "Sensitivity_%", "Specificity_%",
            "Precision_%", "TP", "FP", "TN", "FN", "Duration_sec"
        ]
        return d[keep]

    tidy = pd.concat([build_one(qsvc_map), build_one(vqc_map)], ignore_index=True)
    tidy["k"] = pd.to_numeric(tidy["k"], errors="coerce").astype(int)
    return tidy.reset_index(drop=True)

def _best_per_k_model(tidy: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (k, model), g in tidy.groupby(["k", "Model"], dropna=True):
        g = g.copy()
        for c in ["F1_%", "AUC_%", "Test_Acc_%"]:
            if c not in g.columns:
                g[c] = np.nan
        g = g.sort_values(
            ["F1_%", "AUC_%", "Test_Acc_%"],
            ascending=[False, False, False],
            kind="mergesort"
        )
        rows.append(g.iloc[0])
    return pd.DataFrame(rows).sort_values(["k", "Model"]).reset_index(drop=True)

def _wilson_ci(successes, total, z: float = 1.96):
    """95% Wilson CI for a binomial proportion. Returns (lo%, hi%)."""
    if total is None or total <= 0 or not np.isfinite(total) or not np.isfinite(successes):
        return (np.nan, np.nan)
    k = float(successes)
    n = float(total)
    p = k / n
    denom = 1.0 + (z ** 2) / n
    center = (p + (z ** 2) / (2 * n)) / denom
    half = (z / denom) * np.sqrt((p * (1 - p) / n) + (z ** 2) / (4 * n ** 2))
    return 100.0 * (center - half), 100.0 * (center + half)

# ---------- FNR / FPR plot ----------

def plot_qsvc_fnr_fpr(
    wide_csv: str = WIDE_CSV,
    k_list = (4, 7, 11, 17),
    show: bool = True,
    save: Optional[dict] = None
) -> plt.Figure:

    df_wide = pd.read_csv(wide_csv)
    tidy = _wide_to_tidy(df_wide)
    best = _best_per_k_model(tidy)

    # best QSVC rows for requested k
    qsvc = (
        best[best["Model"] == "QSVC"]
        .set_index("k")
        .loc[list(k_list)]
        .copy()
    )

    ks = np.array(k_list, dtype=int)

    TP = qsvc["TP"].to_numpy(float)
    FP = qsvc["FP"].to_numpy(float)
    TN = qsvc["TN"].to_numpy(float)
    FN = qsvc["FN"].to_numpy(float)

    P = TP + FN
    N = TN + FP

    FNR = np.where(P > 0, 100.0 * FN / P, np.nan)
    FPR = np.where(N > 0, 100.0 * FP / N, np.nan)

    # Wilson CIs
    fn_lo, fn_hi, fp_lo, fp_hi = [], [], [], []
    for fn, p, fp, n in zip(FN, P, FP, N):
        l1, h1 = _wilson_ci(fn, p)  # FNR CI
        l2, h2 = _wilson_ci(fp, n)  # FPR CI
        fn_lo.append(l1); fn_hi.append(h1)
        fp_lo.append(l2); fp_hi.append(h2)
    fn_lo, fn_hi = np.array(fn_lo), np.array(fn_hi)
    fp_lo, fp_hi = np.array(fp_lo), np.array(fp_hi)

    # palette aligned with other figures
    COL_FNR = "#d62728"   # red
    COL_FPR = "#1d5fa7"   # QSVC blue

    # --- figure size for your request ---
    fig, ax = plt.subplots(figsize=(9, 5), dpi=320, constrained_layout=True)

    # small horizontal offset so FNR & FPR (and CIs) are separated
    x_jitter = 0.12
    x_fnr = ks - x_jitter
    x_fpr = ks + x_jitter

    # FNR with CI
    yerr_fnr = np.vstack([FNR - fn_lo, fn_hi - FNR])
    ax.errorbar(
        x_fnr, FNR, yerr=yerr_fnr,
        fmt="-v", color=COL_FNR, lw=2.0, ms=5,
        ecolor=COL_FNR, elinewidth=0.9, capsize=3,
        alpha=0.85, label="FNR = FN / (TP+FN)", zorder=3
    )

    # FPR with CI
    yerr_fpr = np.vstack([FPR - fp_lo, fp_hi - FPR])
    ax.errorbar(
        x_fpr, FPR, yerr=yerr_fpr,
        fmt="-o", color=COL_FPR, lw=2.0, ms=5,
        ecolor=COL_FPR, elinewidth=0.9, capsize=3,
        alpha=0.85, label="FPR = FP / (TN+FP)", zorder=3
    )

    # annotate minima a bit above the corresponding points
    if np.isfinite(FNR).any():
        i_min_fnr = int(np.nanargmin(FNR))
        ax.annotate(
            f"{FNR[i_min_fnr]:.2f}%",
            (x_fnr[i_min_fnr], FNR[i_min_fnr]),
            textcoords="offset points", xytext=(0, 8),
            ha="center", va="bottom", fontsize=9, color=COL_FNR,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8)
        )
    if np.isfinite(FPR).any():
        i_min_fpr = int(np.nanargmin(FPR))
        ax.annotate(
            f"{FPR[i_min_fpr]:.2f}%",
            (x_fpr[i_min_fpr], FPR[i_min_fpr]),
            textcoords="offset points", xytext=(0, 8),
            ha="center", va="bottom", fontsize=9, color=COL_FPR,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.8)
        )

    # axes styling (show full CI range)
    ax.set_xlabel("k (features/qubits)")
    ax.set_ylabel("Rate (%)")
    ax.set_xticks(ks)
    ax.set_xlim(min(ks) - 0.6, max(ks) + 0.6)

    # include full CI in y-limits
    all_lo = np.nanmin([fn_lo.min(), fp_lo.min(), 0.0])
    all_hi = np.nanmax([fn_hi.max(), fp_hi.max()])
    margin = 0.3
    ax.set_ylim(max(0.0, all_lo - margin), all_hi + margin)

    # requested grid style
    ax.yaxis.grid(True, alpha=0.25)
    ax.xaxis.grid(False)

    ax.legend(loc="upper right", frameon=True, fontsize=9)
    ax.set_title("QSVC error rates by k (scaled inputs)")

    if save:
        if "png" in save:
            fig.savefig(save["png"], dpi=320, bbox_inches="tight")
        if "pdf" in save:
            fig.savefig(save["pdf"], dpi=320, bbox_inches="tight")

    if show:
        plt.show()

    return fig

# ---- run directly ----
if __name__ == "__main__":
    plot_qsvc_fnr_fpr(
        wide_csv=WIDE_CSV,
        k_list=(4, 7, 11, 17),
        show=True,
        save=None
    )


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Optional, List

# ---------- helpers ----------
def _is_normalization(s: pd.Series) -> pd.Series:
    low = s.astype(str).str.lower().str.strip()
    # accept exact 'normalization' or any text containing 'normaliz'
    keep = (low == "normalization") | (low.str.contains("normaliz"))
    # exclude obvious unscaled
    keep &= ~low.isin({"none","unscaled","raw"})
    return keep

def _ensure_percent(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().le(1.0 + 1e-12).mean() >= 0.95:
        return s * 100.0
    return s

def _first_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None

def _load_table3_best_if_available() -> Optional[pd.DataFrame]:
    for path in ["tables/table3_best_configs.csv", "table3_best_configs.csv"]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            if "k" not in df.columns and "n_features" in df.columns:
                df = df.rename(columns={"n_features":"k"})
            return df
    return None

def _derive_best_qsvc_from_results(results_csv: str, k_list: List[int]) -> pd.DataFrame:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    # filter scaling = Normalization if present
    if "Scaling" in df.columns:
        df = df[_is_normalization(df["Scaling"])].copy()

    # long vs wide handling
    if "Model" in df.columns:
        d = df[df["Model"].astype(str).str.upper() == "QSVC"].copy()
        for m in ["Test_Acc","AUC","F1","Overfit"]:
            col = _first_col(d, [m, f"{m}_%"])
            if col:
                d[col] = _ensure_percent(d[col])
    else:
        d = df.copy()
        for src in ["QSVC_Test_Acc","QSVC_AUC","QSVC_F1","QSVC_Overfit"]:
            if src in d.columns:
                d[src] = _ensure_percent(d[src])

    out = []
    for k in k_list:
        kval = pd.to_numeric(d.get("k", d.get("n_features")), errors="coerce")
        dk = d[kval == k].copy()
        if dk.empty:
            continue

        test_col = _first_col(dk, ["Test_Acc_%","Test_Acc","QSVC_Test_Acc"])
        auc_col  = _first_col(dk, ["AUC_%","AUC","QSVC_AUC"])
        f1_col   = _first_col(dk, ["F1_%","F1","QSVC_F1"])
        ov_col   = _first_col(dk, ["Overfit_%","Overfit","QSVC_Overfit"])
        dur_col  = _first_col(dk, ["Duration_sec"])

        dk["overfit_abs"] = dk[ov_col].abs() if ov_col else np.inf

        sort_cols, ascending = [], []
        if test_col: sort_cols.append(test_col); ascending.append(False)
        if auc_col:  sort_cols.append(auc_col);  ascending.append(False)
        if f1_col:   sort_cols.append(f1_col);   ascending.append(False)
        sort_cols.append("overfit_abs"); ascending.append(True)
        if dur_col:  sort_cols.append(dur_col);  ascending.append(True)

        best = dk.sort_values(sort_cols, ascending=ascending, kind="mergesort").iloc[0].copy()
        best["k"] = k
        out.append(best)

    best_qsvc = pd.DataFrame(out)

    # pull confusion counts (TP/FP/TN/FN may be in long: TP, or wide: QSVC_TP)
    def _pull(name):
        col = _first_col(best_qsvc, [name, f"QSVC_{name}"])
        return pd.to_numeric(best_qsvc[col], errors="coerce") if col else pd.Series(np.nan, index=best_qsvc.index)

    out_df = pd.DataFrame({
        "k": pd.to_numeric(best_qsvc["k"], errors="coerce").astype("Int64"),
        "Feature_Map": best_qsvc.get("Feature_Map"),
        "Entanglement": best_qsvc.get("Entanglement"),
        "Reps": pd.to_numeric(best_qsvc.get("Reps"), errors="coerce").astype("Int64"),
        "TP": _pull("TP").astype("Int64"),
        "FP": _pull("FP").astype("Int64"),
        "TN": _pull("TN").astype("Int64"),
        "FN": _pull("FN").astype("Int64"),
        "Scaling": best_qsvc.get("Scaling"),
    })
    out_df["N"] = (out_df[["TP","FP","TN","FN"]]
                   .apply(pd.to_numeric, errors="coerce")
                   .sum(axis=1)
                   .astype("Int64"))
    return out_df.sort_values("k").reset_index(drop=True)

def make_table5_qsvc_confusion(
    results_csv: str = "quantum_each_results.csv",
    table3_csv: Optional[str] = "tables/table3_best_configs.csv",
    k_list: List[int] = [4,7,11,17],
    require_normalization: bool = True
) -> pd.DataFrame:
    """
    Returns a DataFrame:
    ['k','Feature_Map','Entanglement','Reps','TP','FP','TN','FN','N','Scaling']
    (QSVC only, scaled 'Normalization'). Prints a plain text table (no tabulate).
    """
    # Prefer an existing best-configs table if present
    df_best = None
    if table3_csv and os.path.exists(table3_csv):
        df_best = pd.read_csv(table3_csv)
    else:
        df_best = _load_table3_best_if_available()

    if df_best is not None:
        df_best.columns = df_best.columns.str.strip()
        if "k" not in df_best.columns and "n_features" in df_best.columns:
            df_best = df_best.rename(columns={"n_features":"k"})
        df_best = df_best[df_best.get("Model","QSVC").astype(str).str.upper() == "QSVC"].copy()
        df_best = df_best[df_best["k"].isin(k_list)].copy()
        if require_normalization and "Scaling" in df_best.columns:
            df_best = df_best[_is_normalization(df_best["Scaling"])].copy()

        cols = ["k","Feature_Map","Entanglement","Reps","TP","FP","TN","FN","Scaling"]
        for c in cols:
            if c not in df_best.columns:
                df_best[c] = np.nan
        out = df_best[cols].copy()
        for c in ["TP","FP","TN","FN","Reps"]:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
        out["N"] = (out[["TP","FP","TN","FN"]]
                    .apply(pd.to_numeric, errors="coerce")
                    .sum(axis=1)
                    .astype("Int64"))
        out = out[["k","Feature_Map","Entanglement","Reps","TP","FP","TN","FN","N","Scaling"]] \
                 .sort_values("k").reset_index(drop=True)
        # print without markdown/tabulate
        print(out.to_string(index=False))
        return out

    # derive from raw results
    out = _derive_best_qsvc_from_results(results_csv, k_list=k_list)
    if require_normalization and "Scaling" in out.columns:
        out = out[_is_normalization(out["Scaling"])].copy()
    out = out[["k","Feature_Map","Entanglement","Reps","TP","FP","TN","FN","N","Scaling"]] \
            .sort_values("k").reset_index(drop=True)
    print(out.to_string(index=False))
    return out

# ---------- optional: stacked bars with NO TITLE ----------
def plot_qsvc_fp_fn_bars_no_title(table5_df: pd.DataFrame):
    """
    Draw stacked bars of FN (bottom, emphasized) + FP (top, lighter) for QSVC.
    No title. A small note is added under the x-axis. Does not save.
    """
    df = table5_df.copy()
    df = df.sort_values("k")
    x = df["k"].astype(int).to_numpy()
    FN = pd.to_numeric(df["FN"], errors="coerce").fillna(0).to_numpy()
    FP = pd.to_numeric(df["FP"], errors="coerce").fillna(0).to_numpy()

    fig, ax = plt.subplots(figsize=(6, 4), dpi=320, constrained_layout=True)
    ax.bar(x, FN, label="FN", color="#d62728", alpha=0.85)           # red (emphasized)
    ax.bar(x, FP, bottom=FN, label="FP", color="#ff9896", alpha=0.85) # lighter red/pink

    # annotations
    for xi, fn, fp in zip(x, FN, FP):
        if fn > 0:
            ax.text(xi, fn/2, f"{int(fn)}", ha="center", va="center", color="white", fontsize=10)
        if fp > 0:
            ax.text(xi, fn + fp/2, f"{int(fp)}", ha="center", va="center", color="black", fontsize=10)

    ax.set_xlabel("k (features/qubits)", fontsize=11)
    ax.set_ylabel("Count (test set)", fontsize=11)
    ax.set_xticks(x)
    ax.grid(axis="y", alpha=0.2)
    # NO title per your request
    # add note below x-axis
    fig.text(0.5, -0.02, "In screening, minimize FN", ha="center", va="top", fontsize=10)
    ax.legend(loc="upper right", frameon=False)
    plt.show()

# -------- run: build Table 5 (prints only) --------
table5 = make_table5_qsvc_confusion(
    results_csv="quantum_each_results.csv",
    table3_csv="tables/table3_best_configs.csv",  # change or set None if you want to derive from raw
    k_list=[4,7,11,17],
    require_normalization=True
)

# If you also want the stacked bars without a title, uncomment:
# plot_qsvc_fp_fn_bars_no_title(table5)



In [ ]:
import os, json
from pathlib import Path
from typing import Optional, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE
from sklearn.metrics import (
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    confusion_matrix, matthews_corrcoef, accuracy_score
)

# Qiskit (same stack you used)
from qiskit.circuit.library import ZFeatureMap, PauliFeatureMap
from qiskit_machine_learning.kernels import FidelityStatevectorKernel
from qiskit_machine_learning.algorithms.classifiers import QSVC

np.random.seed(42)

# ----------------------------- fixed best QSVC configs -----------------------------
BEST_CFGS = [
    dict(k=4,  fmap="ZFeatureMap",     ent="full",     reps=1),
    dict(k=7,  fmap="PauliFeatureMap", ent="circular", reps=4),
    dict(k=11, fmap="PauliFeatureMap", ent="linear",   reps=4),
    dict(k=17, fmap="ZFeatureMap",     ent="full",     reps=1),
]

# --------------------------------- helpers -----------------------------------------
def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def _load_table3_qsvc(table3_csv: str) -> pd.DataFrame:
    df = pd.read_csv(table3_csv)
    req = {"k","Model","Feature_Map","Entanglement","Reps","Scaling"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Table3 missing columns: {sorted(missing)}")
    keep_k = {c["k"] for c in BEST_CFGS}
    d = df[(df["Model"].astype(str).str.upper()=="QSVC") & (df["k"].isin(keep_k))].copy()
    if d.empty:
        raise ValueError("No QSVC rows for k∈{4,7,11,17} found in Table 3 CSV.")
    return d

def _load_dataset(data_csv: Optional[str], target: str, pos_label: Optional[str]) -> Tuple[pd.DataFrame, pd.Series]:
    """
    If data_csv is None -> use sklearn WBCD dataset directly.
    Otherwise use the provided CSV (map positive label to 1).
    """
    if data_csv:
        df = pd.read_csv(data_csv)
        if target not in df.columns:
            raise ValueError(f"Target column '{target}' not in {data_csv}.")
        y_raw = df[target].values
        X = df.drop(columns=[target])
        if pos_label is not None:
            y = (y_raw == pos_label).astype(int)
        else:
            y = pd.to_numeric(y_raw, errors="coerce").fillna(0).astype(int)
        return X.reset_index(drop=True), pd.Series(y, name="y").reset_index(drop=True)
    else:
        # fallback: scikit-learn WBCD
        from sklearn.datasets import load_breast_cancer
        data = load_breast_cancer()
        X = pd.DataFrame(data.data, columns=data.feature_names)
        y = pd.Series(data.target, name="y")
        return X, y

def _load_or_make_split(y: pd.Series, split_path: str = "splits/split_idx.json", test_size: float = 0.2):
    split_path = Path(split_path); _ensure_dir(split_path.parent)
    if split_path.exists():
        with open(split_path, "r", encoding="utf-8") as f:
            j = json.load(f)
        return np.array(j["train_idx"], dtype=int), np.array(j["test_idx"], dtype=int)
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
    (train_idx, test_idx), = sss.split(np.zeros(len(y), dtype=int), y.values.astype(int))
    with open(split_path, "w", encoding="utf-8") as f:
        json.dump({"train_idx":train_idx.tolist(), "test_idx":test_idx.tolist()}, f, indent=2)
    return train_idx, test_idx

def _read_feature_list_from_dir(features_dir: str, k: int) -> Optional[List[str]]:
    base = Path(features_dir)
    candidates = [
        base / f"selected_k{k}.json",
        base / f"selected_k{k}.csv",
        base / f"selected_k{k}.txt",
    ]
    for p in candidates:
        if p.suffix == ".json" and p.exists():
            with open(p, "r", encoding="utf-8") as f:
                obj = json.load(f)
            if isinstance(obj, dict) and "columns" in obj:
                return list(obj["columns"])
            if isinstance(obj, list):
                return list(obj)
            raise ValueError(f"Unsupported JSON structure in {p}.")
        if p.suffix == ".csv" and p.exists():
            df = pd.read_csv(p)
            if "feature" in df.columns:
                return df["feature"].astype(str).tolist()
            return df.iloc[:,0].astype(str).tolist()
        if p.suffix == ".txt" and p.exists():
            with open(p, "r", encoding="utf-8") as f:
                return [line.strip() for line in f if line.strip()]
    return None

def _read_feature_list_fallback_rfe_csv(rfe_csv: str, k: int) -> Optional[List[str]]:
    p = Path(rfe_csv)
    if not p.exists():
        return None
    try:
        df = pd.read_csv(p, converters={"Selected Features": pd.eval})
        row = df[df["n_features"] == k].head(1)
        if not row.empty:
            return list(row.iloc[0]["Selected Features"])
    except Exception:
        return None
    return None

def _compute_rfe_cols(X_train: pd.DataFrame, y_train: np.ndarray, k: int) -> List[str]:
    lr = LogisticRegression(max_iter=1000, random_state=42)
    rfe = RFE(lr, n_features_to_select=k)
    rfe.fit(X_train, y_train)
    mask = rfe.support_
    return X_train.columns[mask].tolist()

def _build_feature_map(kind: str, k: int, ent: str, reps: int):
    # ZFeatureMap: no entanglement arg (to match many Qiskit versions)
    if kind == "ZFeatureMap":
        return ZFeatureMap(feature_dimension=k, reps=int(reps))
    elif kind == "PauliFeatureMap":
        return PauliFeatureMap(feature_dimension=k, reps=int(reps), entanglement=ent)
    else:
        raise ValueError(f"Unsupported Feature_Map: {kind}")

def _metrics_from_scores(y_true: np.ndarray, y_score: np.ndarray, y_pred: np.ndarray) -> Dict:
    cm = confusion_matrix(y_true, y_pred, labels=[1,0])  # [[TP,FN],[FP,TN]]
    tp, fn = cm[0,0], cm[0,1]
    fp, tn = cm[1,0], cm[1,1]
    acc = accuracy_score(y_true, y_pred)
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    f1   = (2*prec*sens)/(prec+sens) if (prec+sens) else 0.0
    mcc  = matthews_corrcoef(y_true, y_pred)

    fpr, tpr, thr = roc_curve(y_true, y_score)
    rocA = roc_auc_score(y_true, y_score)

    prec_curve, rec_curve, thr_pr = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    F1s = 2*prec_curve[:-1]*rec_curve[:-1] / np.where((prec_curve[:-1]+rec_curve[:-1])==0, 1, (prec_curve[:-1]+rec_curve[:-1]))
    i_best = int(np.nanargmax(F1s)) if F1s.size else 0
    F1_max = float(F1s[i_best]) if F1s.size else float(f1)
    thr_best = float(thr_pr[i_best]) if thr_pr.size else 0.0

    if thr_pr.size:
        y_pred_best = (y_score >= thr_best).astype(int)
        mcc_best = matthews_corrcoef(y_true, y_pred_best)
    else:
        mcc_best = mcc

    return dict(
        TP=int(tp), FP=int(fp), TN=int(tn), FN=int(fn),
        accuracy=float(acc), sensitivity=float(sens), specificity=float(spec),
        precision=float(prec), F1=float(f1), mcc=float(mcc),
        fpr=fpr, tpr=tpr, thr=thr,
        recall=rec_curve, precision_curve=prec_curve, thr_pr=thr_pr,
        roc_auc=float(rocA), pr_auc=float(ap),
        F1_max=float(F1_max), thr_F1_max=float(thr_best), MCC_at_F1_max=float(mcc_best),
    )

# ----------------------------------- core ------------------------------------------
def export_qsvc_rocpr_from_table3(
    table3_csv: str = "tables/Table3_best_configs.csv",
    features_dir: str = "Selected Features",
    data_csv: Optional[str] = None,              # <-- use WBCD by default
    target: str = "diagnosis",
    pos_label: str = "M",
    save_dir: str = "outputs/rocpr_qsvc",
    split_path: str = "splits/split_idx.json",
    test_size: float = 0.2,
    scale: str = "normalizer",
    force: bool = False,
    rfe_fallback_csv: str = "classical_rfe_fixed_counts.csv",
):
    """
    Re-run only QSVC best configs from Table 3 for k=4,7,11,17 and export per-sample
    scores, ROC/PR curves, and metrics. Checkpoint after each k.
    Uses sklearn WBCD if data_csv is None.
    """
    outdir = Path(save_dir); _ensure_dir(outdir)

    # 1) Load Table 3 and filter
    tbl = _load_table3_qsvc(table3_csv)
    keep_k = {4,7,11,17}
    tbl = tbl[tbl["k"].isin(keep_k)].copy()

    # 2) Data + split
    X, y = _load_dataset(data_csv, target, pos_label)
    train_idx, test_idx = _load_or_make_split(y, split_path, test_size)

    # 3) Loop over fixed best configs (controlled order)
    rows_summary = []
    for cfg in BEST_CFGS:
        k = cfg["k"]
        row = tbl[tbl["k"]==k].head(1)
        if row.empty:
            print(f"[SKIP] k={k} not present in Table 3 QSVC filter.")
            continue
        fmap = str(row.iloc[0]["Feature_Map"])
        ent  = str(row.iloc[0]["Entanglement"])
        reps = int(row.iloc[0]["Reps"])

        # resume logic
        metrics_json = outdir / f"metrics_k{k}.json"
        scores_csv   = outdir / f"scores_k{k}.csv"
        roc_csv      = outdir / f"roc_k{k}.csv"
        pr_csv       = outdir / f"pr_k{k}.csv"
        if (metrics_json.exists() and scores_csv.exists() and roc_csv.exists() and pr_csv.exists()) and (not force):
            print(f"[OK] k={k} already exported. Use force=True to recompute.")
            with open(metrics_json, "r", encoding="utf-8") as f:
                m = json.load(f)
            rows_summary.append(dict(
                k=k, roc_auc=m["roc_auc"], pr_auc=m["pr_auc"], mcc=m["mcc"],
                accuracy=m["accuracy"], sensitivity=m["sensitivity"],
                specificity=m["specificity"], precision=m["precision"],
                F1=m["F1"], TP=m["TP"], FP=m["FP"], TN=m["TN"], FN=m["FN"]
            ))
            continue

        # 4) Features for this k (try directory -> RFE CSV -> compute RFE)
        feat_cols = _read_feature_list_from_dir(features_dir, k)
        if feat_cols is None:
            feat_cols = _read_feature_list_fallback_rfe_csv(rfe_fallback_csv, k)
        if feat_cols is None:
            # compute via RFE on training split (scaled as per 'scale' option)
            if scale.lower().startswith("standard"):
                scal = StandardScaler().fit(X.iloc[train_idx, :].values)
                X_train_scaled = pd.DataFrame(scal.transform(X.iloc[train_idx, :].values),
                                              columns=X.columns)
            else:
                scal = Normalizer(norm="l2").fit(X.iloc[train_idx, :].values)
                X_train_scaled = pd.DataFrame(scal.transform(X.iloc[train_idx, :].values),
                                              columns=X.columns)
            feat_cols = _compute_rfe_cols(X_train_scaled, y.iloc[train_idx].values.astype(int), k)
            print(f"[INFO] Computed RFE features for k={k}: {len(feat_cols)} columns.")

        # subset
        X_train = X.iloc[train_idx][feat_cols].copy()
        X_test  = X.iloc[test_idx][feat_cols].copy()

        # 5) Scaling: Normalizer by default (as per Table 3 "Normalization")
        if scale.lower().startswith("standard"):
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train.values)
            X_test_s  = scaler.transform(X_test.values)
        else:
            scaler = Normalizer(norm="l2")
            X_train_s = scaler.fit_transform(X_train.values)
            X_test_s  = scaler.transform(X_test.values)

        # 6) QSVC model
        fmap_circ = _build_feature_map(fmap, k, ent, reps)
        kernel = FidelityStatevectorKernel(feature_map=fmap_circ)
        clf = QSVC(quantum_kernel=kernel)
        clf.fit(X_train_s, y.iloc[train_idx].values.astype(int))

        # 7) Scores/preds
        y_true = y.iloc[test_idx].values.astype(int)
        try:
            y_score = clf.decision_function(X_test_s).ravel()
        except Exception:
            if hasattr(clf, "predict_proba"):
                y_score = clf.predict_proba(X_test_s)[:,1]
            else:
                y_score = clf.predict(X_test_s).astype(float)
        y_pred = clf.predict(X_test_s).astype(int)

        # 8) Metrics & curves
        mets = _metrics_from_scores(y_true, y_score, y_pred)

        # 9) Save per-k artifacts (checkpoint)
        pd.DataFrame({"y_true":y_true, "y_score":y_score, "y_pred":y_pred, "k":k}).to_csv(scores_csv, index=False)
        pd.DataFrame({"fpr":mets["fpr"], "tpr":mets["tpr"], "threshold":mets["thr"]}).to_csv(roc_csv, index=False)
        thr_pr = mets["thr_pr"]
        if thr_pr.size and thr_pr.size < mets["recall"].size:
            thr_pr = np.append(thr_pr, np.nan)
        pd.DataFrame({"recall":mets["recall"], "precision":mets["precision_curve"], "threshold":thr_pr}).to_csv(pr_csv, index=False)

        payload = {
            "k":k, "Feature_Map":fmap, "Entanglement":ent, "Reps":reps, "scale":scale,
            "TP":mets["TP"], "FP":mets["FP"], "TN":mets["TN"], "FN":mets["FN"],
            "accuracy":mets["accuracy"], "sensitivity":mets["sensitivity"], "specificity":mets["specificity"],
            "precision":mets["precision"], "F1":mets["F1"], "mcc":mets["mcc"],
            "roc_auc":mets["roc_auc"], "pr_auc":mets["pr_auc"],
            "F1_max":mets["F1_max"], "thr_F1_max":mets["thr_F1_max"], "MCC_at_F1_max":mets["MCC_at_F1_max"]
        }
        with open(metrics_json, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)

        print(f"k={k} | ROC-AUC={mets['roc_auc']:.3f} | PR-AUC={mets['pr_auc']:.3f} | "
              f"MCC={mets['mcc']:.3f} | TP={mets['TP']} FP={mets['FP']} TN={mets['TN']} FN={mets['FN']} "
              f"(saved to {outdir}/)")

        rows_summary.append(dict(
            k=k, roc_auc=mets["roc_auc"], pr_auc=mets["pr_auc"], mcc=mets["mcc"],
            accuracy=mets["accuracy"], sensitivity=mets["sensitivity"],
            specificity=mets["specificity"], precision=mets["precision"],
            F1=mets["F1"], TP=mets["TP"], FP=mets["FP"], TN=mets["TN"], FN=mets["FN"]
        ))

    # 10) Summary CSV
    if rows_summary:
        df_sum = pd.DataFrame(rows_summary).sort_values("k")
        df_sum.to_csv(outdir / "summary.csv", index=False)

    # 11) Figures (read back CSVs so resume works)
    ks_done = [c["k"] for c in BEST_CFGS if (outdir / f"scores_k{c['k']}.csv").exists()]
    if not ks_done:
        print("[WARN] No per-k scores found; skipping figures.")
        return

    # ROC
    plt.figure(figsize=(7.4, 5.3), dpi=320)
    plt.plot([0,1],[0,1], "--", color="#cccccc", lw=1)
    color_map = {4:"tab:blue", 7:"tab:orange", 11:"tab:green", 17:"tab:red"}
    for k in ks_done:
        sc = pd.read_csv(outdir / f"scores_k{k}.csv")
        fpr, tpr, _ = roc_curve(sc["y_true"].values, sc["y_score"].values)
        auc = roc_auc_score(sc["y_true"].values, sc["y_score"].values)
        plt.plot(fpr, tpr, lw=2, color=color_map.get(k, None), label=f"k={k} (AUC={auc:.3f})")
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title("QSVC ROC (scaled, best per k)"); plt.grid(alpha=0.2)
    plt.legend(loc="lower right", frameon=True, fontsize=9)
    plt.tight_layout()
    plt.savefig(outdir / "fig_roc_qsvc_best.pdf", bbox_inches="tight")
    plt.savefig(outdir / "fig_roc_qsvc_best.png", dpi=300, bbox_inches="tight")
    plt.show()

    # PR
    plt.figure(figsize=(7.4, 5.3), dpi=320)
    sc0 = pd.read_csv(outdir / f"scores_k{ks_done[0]}.csv")
    base = float(sc0["y_true"].mean())
    plt.plot([0,1],[base, base], "--", color="#cccccc", lw=1, label="prevalence")
    for k in ks_done:
        sc = pd.read_csv(outdir / f"scores_k{k}.csv")
        prec, rec, _ = precision_recall_curve(sc["y_true"].values, sc["y_score"].values)
        ap = average_precision_score(sc["y_true"].values, sc["y_score"].values)
        plt.plot(rec, prec, lw=2, color=color_map.get(k, None), label=f"k={k} (AP={ap:.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("QSVC Precision–Recall (scaled, best per k)"); plt.grid(alpha=0.2)
    plt.legend(loc="lower left", frameon=True, fontsize=9)
    plt.tight_layout()
    plt.savefig(outdir / "fig_pr_qsvc_best.pdf", bbox_inches="tight")
    plt.savefig(outdir / "fig_pr_qsvc_best.png", dpi=300, bbox_inches="tight")
    plt.show()


# ---------------------------- run it (WBCD default) ----------------------------
export_qsvc_rocpr_from_table3(
    table3_csv="tables/Table3_best_configs.csv",
    features_dir="Selected Features",
    data_csv=None,                 # <-- WBCD fallback (no external CSV)
    target="diagnosis",            # ignored when data_csv=None
    pos_label="M",                 # ignored when data_csv=None
    save_dir="outputs/rocpr_qsvc",
    split_path="splits/split_idx.json",
    test_size=0.2,
    scale="normalizer",            # or "standard"
    force=False,                   # set True to recompute even if artifacts exist
    rfe_fallback_csv="classical_rfe_fixed_counts.csv"
)


## Geometry and feature-space exploration

These cells focus on sphere-based visualizations and feature selection follow-up around the best-performing configurations.

In [ ]:
# ===== Compare EXPLICIT (QSVC) vs IMPLICIT (VQC) — Best 4-feature configs =====
# - Picks best QSVC (by QSVC_AUC → QSVC_Test_Acc) with n_features == 4
# - Picks best VQC  (by VQC_AUC  → VQC_Test_Acc)  with n_features == 4
# - Applies each row's Scaling and EXACT 4 features (if present in CSV)
# - Plots 2x2: [QSVC BEFORE | QSVC AFTER] vs [VQC BEFORE | VQC AFTER]
#
# NOTE: "AFTER" uses an SVM on the shown sphere points as a decision proxy.
#       Swap in real per-sample decision values if you saved them.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC

# ======== USER SETTINGS ========
CSV_PATH = "quantum_each_results.csv"
TARGET_N_FEATURES = 7
CAMERA_ELEV, CAMERA_AZIM = 20, 45     # rotate if you prefer
POINT_SIZE = 14
FEATURE_LIST_CANDIDATES = ["Selected_Features", "Features", "Feature_List", "Top_Features"]
# ===============================

def parse_metric(x):
    if pd.isna(x): return np.nan
    if isinstance(x, (int, float)): return float(x)
    s = str(x).strip()
    if s.endswith("%"): s = s[:-1]
    try: return float(s)
    except: return np.nan

def fmt_metric(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return "—"
    return f"{float(x):.2f}"

def pick_best_qsvc_row(df):
    sub = df[df["n_features"].astype(int) == TARGET_N_FEATURES].copy()
    if sub.empty:
        raise ValueError(f"No rows with n_features == {TARGET_N_FEATURES}")
    need = ["QSVC_AUC", "QSVC_Test_Acc"]
    for c in need:
        if c not in sub.columns:
            raise ValueError(f"CSV missing column '{c}' for QSVC selection.")
    sub["_auc"] = sub["QSVC_AUC"].map(parse_metric)
    sub["_acc"] = sub["QSVC_Test_Acc"].map(parse_metric)
    sub = sub.sort_values(["_auc", "_acc"], ascending=False, na_position="last")
    return sub.iloc[0]

def pick_best_vqc_row(df):
    sub = df[df["n_features"].astype(int) == TARGET_N_FEATURES].copy()
    if sub.empty:
        raise ValueError(f"No rows with n_features == {TARGET_N_FEATURES}")
    need = ["VQC_AUC", "VQC_Test_Acc"]
    for c in need:
        if c not in sub.columns:
            raise ValueError(f"CSV missing column '{c}' for VQC selection.")
    sub["_auc"] = sub["VQC_AUC"].map(parse_metric)
    sub["_acc"] = sub["VQC_Test_Acc"].map(parse_metric)
    sub = sub.sort_values(["_auc", "_acc"], ascending=False, na_position="last")
    return sub.iloc[0]

def parse_feature_indices_from_row(row, all_feature_names):
    """Return list of 4 indices if the CSV stores the exact feature names."""
    for col in FEATURE_LIST_CANDIDATES:
        if col in row.index and pd.notna(row[col]):
            raw = str(row[col]).strip().strip("[](){}")
            parts = [p.strip().strip("'").strip('"') for p in raw.split(",")]
            parts = [p for p in parts if p]
            idxs = []
            for name in parts:
                if name in all_feature_names:
                    idxs.append(list(all_feature_names).index(name))
                else:
                    # loose match if needed
                    matches = [i for i, fn in enumerate(all_feature_names) if name.lower() in fn.lower()]
                    if matches: idxs.append(matches[0])
            if idxs:
                return sorted(set(idxs))
    return None

def get_scaler_from_text(txt):
    s = str(txt).lower()
    # your CSV uses "Normalization" and "Standardization" naming;
    # choose MinMax for normalization; Standard for others by default.
    return MinMaxScaler() if "norm" in s or "min" in s else StandardScaler()

def to_unit_sphere(Xmat):
    """Reduce to 3D then normalize each row to the unit sphere (for plotting)."""
    pca = PCA(n_components=min(3, Xmat.shape[1]), random_state=0)
    Z = pca.fit_transform(Xmat)
    if Z.shape[1] < 3:
        Z = np.pad(Z, ((0,0),(0, 3-Z.shape[1])), mode="constant")
    R = np.linalg.norm(Z, axis=1, keepdims=True); R[R==0] = 1.0
    return Z / R

def draw_wire_sphere(ax):
    u = np.linspace(0, 2*np.pi, 40)
    v = np.linspace(0, np.pi, 20)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, linewidth=0.25, alpha=0.25)

def show_points(ax, P, labels=None, title=None, support_mask=None, color_values=None):
    if color_values is None:
        if labels is None:
            ax.scatter(P[:,0], P[:,1], P[:,2], s=POINT_SIZE, alpha=0.9)
        else:
            m0 = (labels == 0); m1 = ~m0
            ax.scatter(P[m0,0], P[m0,1], P[m0,2], s=POINT_SIZE, alpha=0.9, label="Class 0")
            ax.scatter(P[m1,0], P[m1,1], P[m1,2], s=POINT_SIZE, alpha=0.9, label="Class 1")
            ax.legend(loc="upper left", fontsize=8)
    else:
        sc = ax.scatter(P[:,0], P[:,1], P[:,2], c=color_values, cmap="viridis",
                        s=POINT_SIZE, alpha=0.95)
        cb = plt.colorbar(sc, ax=ax, shrink=0.8, pad=0.05)
        cb.set_label("Decision value (normalized)")
    if support_mask is not None:
        ax.scatter(P[support_mask,0], P[support_mask,1], P[support_mask,2],
                   s=POINT_SIZE+30, facecolors='none', edgecolors='k', linewidths=1.0,
                   label="Support Vectors")
    ax.set_box_aspect([1,1,1])
    ax.set_xlim([-1,1]); ax.set_ylim([-1,1]); ax.set_zlim([-1,1])
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    if title: ax.set_title(title, pad=8)

# ---------- Load CSV and pick best rows ----------
df = pd.read_csv(CSV_PATH)

best_qsvc = pick_best_qsvc_row(df)
best_vqc  = pick_best_vqc_row(df)

print("Best QSVC row (n_features=4):")
display(best_qsvc.to_frame().T)
print("Best VQC row (n_features=4):")
display(best_vqc.to_frame().T)

# ---------- Load WBCD ----------
data = load_breast_cancer()
X_full = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

# ---------- Build QSVC view ----------
qsvc_scaler = get_scaler_from_text(best_qsvc.get("Scaling", "standard"))
X_scaled_qsvc = qsvc_scaler.fit_transform(X_full.values)

qsvc_feat_idx = parse_feature_indices_from_row(best_qsvc, data.feature_names)
Xq_use = X_scaled_qsvc[:, qsvc_feat_idx] if (qsvc_feat_idx and len(qsvc_feat_idx) >= 1) else X_scaled_qsvc
U_qsvc = to_unit_sphere(Xq_use)

# Decision proxy for AFTER
clf_q = SVC(kernel="rbf", C=1.0, gamma="scale")
clf_q.fit(U_qsvc, y.values)
dec_q = clf_q.decision_function(U_qsvc)
sv_mask_q = np.zeros(U_qsvc.shape[0], dtype=bool); sv_mask_q[clf_q.support_] = True
dmax_q = np.max(np.abs(dec_q)) or 1.0
colors_q = (dec_q / dmax_q + 1) / 2.0

qsvc_title_metrics = f"AUC={fmt_metric(parse_metric(best_qsvc.get('QSVC_AUC')))} | " \
                     f"Acc={fmt_metric(parse_metric(best_qsvc.get('QSVC_Test_Acc')))}"
qsvc_feat_title = ""
if qsvc_feat_idx and len(qsvc_feat_idx) == 4:
    qsvc_feat_title = "Features: " + ", ".join([data.feature_names[i] for i in qsvc_feat_idx])

# ---------- Build VQC view ----------
vqc_scaler = get_scaler_from_text(best_vqc.get("Scaling", "standard"))
X_scaled_vqc = vqc_scaler.fit_transform(X_full.values)

vqc_feat_idx = parse_feature_indices_from_row(best_vqc, data.feature_names)
Xv_use = X_scaled_vqc[:, vqc_feat_idx] if (vqc_feat_idx and len(vqc_feat_idx) >= 1) else X_scaled_vqc
U_vqc = to_unit_sphere(Xv_use)

# Decision proxy for AFTER
clf_v = SVC(kernel="rbf", C=1.0, gamma="scale")
clf_v.fit(U_vqc, y.values)
dec_v = clf_v.decision_function(U_vqc)
sv_mask_v = np.zeros(U_vqc.shape[0], dtype=bool); sv_mask_v[clf_v.support_] = True
dmax_v = np.max(np.abs(dec_v)) or 1.0
colors_v = (dec_v / dmax_v + 1) / 2.0

vqc_title_metrics = f"AUC={fmt_metric(parse_metric(best_vqc.get('VQC_AUC')))} | " \
                    f"Acc={fmt_metric(parse_metric(best_vqc.get('VQC_Test_Acc')))}"
vqc_feat_title = ""
if vqc_feat_idx and len(vqc_feat_idx) == 4:
    vqc_feat_title = "Features: " + ", ".join([data.feature_names[i] for i in vqc_feat_idx])

# ---------- Plot 2x2 ----------
fig = plt.figure(figsize=(14, 10), dpi=130)

# QSVC BEFORE
ax1 = fig.add_subplot(2,2,1, projection='3d')
ax1.view_init(CAMERA_ELEV, CAMERA_AZIM)
draw_wire_sphere(ax1)
title1 = f"QSVC — BEFORE (by class)\n{qsvc_feat_title}"
show_points(ax1, U_qsvc, labels=y.values, title=title1)

# QSVC AFTER
ax2 = fig.add_subplot(2,2,2, projection='3d')
ax2.view_init(CAMERA_ELEV, CAMERA_AZIM)
draw_wire_sphere(ax2)
title2 = f"QSVC — AFTER (proxy)\n{qsvc_title_metrics}"
show_points(ax2, U_qsvc, title=title2, support_mask=sv_mask_q, color_values=colors_q)

# VQC BEFORE
ax3 = fig.add_subplot(2,2,3, projection='3d')
ax3.view_init(CAMERA_ELEV, CAMERA_AZIM)
draw_wire_sphere(ax3)
title3 = f"VQC — BEFORE (by class)\n{vqc_feat_title}"
show_points(ax3, U_vqc, labels=y.values, title=title3)

# VQC AFTER
ax4 = fig.add_subplot(2,2,4, projection='3d')
ax4.view_init(CAMERA_ELEV, CAMERA_AZIM)
draw_wire_sphere(ax4)
title4 = f"VQC — AFTER (proxy)\n{vqc_title_metrics}"
show_points(ax4, U_vqc, title=title4, support_mask=sv_mask_v, color_values=colors_v)

plt.suptitle("Best (n_features=4) — QSVC (explicit) vs VQC (implicit): BEFORE vs AFTER", fontsize=14)
plt.tight_layout(rect=[0,0,1,0.95])
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import ast

# Optional: joblib for estimator loading (if you have pickled models)
try:
    import joblib
except Exception:
    joblib = None

# ============== helpers ==============

def _percentize(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce")
    if s.dropna().le(1.0 + 1e-12).mean() >= 0.95:  # mostly in [0,1]
        return s * 100.0
    return s

def _is_scaled(s: pd.Series) -> pd.Series:
    low = s.astype(str).str.lower().str.strip()
    keep = (
        low.str.contains("standard") | low.str.contains("std") |
        low.str.contains("minmax")   | low.str.contains("normalize") |
        low.str.contains("normaliz") | low.str.contains("scaled") |
        low.str.contains("robust")
    )
    keep &= ~low.isin({"none","unscaled","raw","nan","na"})
    return keep

def _maybe_eval_list(x):
    """Turn strings like "[1,2]" or "['a','b']" into lists when possible."""
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    if isinstance(x, (list, tuple)):
        return list(x)
    s = str(x).strip()
    if not s or s.lower() in {"none", "nan"}:
        return None
    try:
        v = ast.literal_eval(s)
        if isinstance(v, (list, tuple)):
            return list(v)
        if isinstance(v, str) and "," in v:
            return [p.strip() for p in v.split(",")]
    except Exception:
        if "," in s:
            return [p.strip() for p in s.split(",")]
    return None

def _first_present(row, candidates):
    """Return (value, source_col) for first existing, non-null column in candidates."""
    for col in candidates:
        if col in row.index:
            val = row[col]
            if pd.notna(val):
                return val, col
    return None, None

def _summarize_estimator(est):
    """Tiny, robust summary for common sklearn estimators."""
    summary = {"type": type(est).__name__}
    for attr in ["n_features_in_", "classes_", "coef_", "intercept_"]:
        if hasattr(est, attr):
            val = getattr(est, attr)
            try:
                if hasattr(val, "shape"):
                    summary[attr] = f"shape={tuple(val.shape)}"
                elif isinstance(val, (list, tuple)):
                    summary[attr] = f"len={len(val)}"
                else:
                    summary[attr] = str(val)
            except Exception:
                summary[attr] = "<unrepr>"
    return summary

def _extract_feature_info(row, model=None):
    """
    From one k=7 row, pull selected feature names/indices.
    Tries model-specific columns first, then global.
    """
    cand_names, cand_idx = [], []

    if model:
        cand_names += [f"{model}_Selected_Features", f"{model}_Selected_Names"]
        cand_idx   += [f"{model}_Selected_Idx"]

    cand_names += ["Selected_Features", "Selected_Names"]
    cand_idx   += ["Selected_Idx", "Feature_Idx"]

    names_val, names_col = _first_present(row, cand_names)
    idx_val,   idx_col   = _first_present(row, cand_idx)

    names_list = _maybe_eval_list(names_val)
    idx_list   = _maybe_eval_list(idx_val)
    if idx_list is not None:
        try:
            idx_list = [int(x) for x in idx_list]
        except Exception:
            pass

    # Map indices → names if we have a full list
    if names_list is None and idx_list is not None:
        full_val, _ = _first_present(row, ["All_Feature_Names", "Feature_Names"])
        full_list = _maybe_eval_list(full_val)
        if isinstance(full_list, list) and all(isinstance(i, (int, np.integer)) for i in idx_list):
            try:
                names_list = [full_list[i] for i in idx_list]
            except Exception:
                pass

    return {
        "selected_names": names_list,
        "selected_idx": idx_list,
        "names_col": names_col,
        "idx_col": idx_col
    }

def _extract_estimator_info(row, model="LR", load_estimator=False):
    """Get estimator path/blob; optionally load via joblib."""
    est_val, est_col = _first_present(
        row, [f"{model}_Estimator_Path", f"{model}_Estimator", "Estimator_Path", "Estimator"]
    )
    info = {
        "estimator_source": est_col,
        "estimator_path_or_blob": est_val,
        "loaded": False,
        "summary": None
    }
    if not load_estimator or est_val is None:
        return info
    if isinstance(est_val, str) and joblib is not None:
        try:
            est = joblib.load(est_val)
            info["loaded"] = True
            info["summary"] = _summarize_estimator(est)
        except Exception:
            pass
    return info

def _match_extra_keys(df):
    """Optional keys that help match 'the same run/feature set' across models if present."""
    return [c for c in [
        "Feature_Set_ID", "Run_ID", "Fold", "Seed", "Random_State", "Split_ID"
    ] if c in df.columns]

def _format_list_short(x, max_items=7):
    if not x:
        return ""
    x = list(x)
    if len(x) <= max_items:
        return ", ".join(map(str, x))
    return ", ".join(map(str, x[:max_items])) + f" … (+{len(x)-max_items})"

# ============== wide/long normalizer ==============

def _to_long(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize to long format with columns:
    ['n_features','Model','Feature_Map','Entanglement','Reps','AUC_%','Test_Acc_%','F1_%','Scaling']
    Preserve extra columns (feature/estimator info + run IDs).
    Works for either wide (QSVC_*/VQC_*/LR_*) or long inputs.
    """
    df = df.copy()
    df.columns = df.columns.str.strip()

    if "Scaling" in df.columns:
        df = df[_is_scaled(df["Scaling"])].copy()

    extra_candidates = {
        # features
        "Selected_Features", "Selected_Names", "Selected_Idx", "Feature_Idx",
        "All_Feature_Names", "Feature_Names",
        "LR_Selected_Features", "LR_Selected_Names", "LR_Selected_Idx",
        "QSVC_Selected_Features", "VQC_Selected_Features",
        # estimator
        "Estimator", "Estimator_Path",
        "LR_Estimator", "LR_Estimator_Path",
        "QSVC_Estimator", "VQC_Estimator",
        # matching keys
        "Feature_Set_ID", "Run_ID", "Fold", "Seed", "Random_State", "Split_ID"
    }
    extras_present = [c for c in df.columns if c in extra_candidates]

    if "Model" in df.columns:
        out = df.copy()
        for base_in, base_out in [("AUC","AUC_%"), ("Test_Acc","Test_Acc_%"), ("F1","F1_%")]:
            cand = [c for c in out.columns if c.endswith(base_in)]
            for c in cand:
                out[c] = _percentize(out[c])
                out.rename(columns={c: base_out}, inplace=True)
        for c in ["Feature_Map","Entanglement","Reps","Scaling"]:
            if c not in out.columns:
                out[c] = np.nan
        base_keep = ["n_features","Model","Feature_Map","Entanglement","Reps","AUC_%","Test_Acc_%","F1_%","Scaling"]
        keep = base_keep + [c for c in extras_present if c not in base_keep]
        for c in base_keep:
            if c not in out.columns:
                out[c] = np.nan
        return out[keep].copy()

    # wide → long
    rows = []
    for _, r in df.iterrows():
        base = dict(
            n_features=r.get("n_features", np.nan),
            Feature_Map=r.get("Feature_Map", np.nan),
            Entanglement=r.get("Entanglement", np.nan),
            Reps=r.get("Reps", np.nan),
            Scaling=r.get("Scaling", np.nan),
        )
        for c in extras_present:
            base[c] = r.get(c, np.nan)

        for model in ["QSVC","VQC","LR"]:
            pref    = f"{model}_"
            auc_col = pref + "AUC"
            if auc_col in df.columns:
                rowm = base.copy()
                rowm["Model"]       = model
                rowm["AUC_%"]       = _percentize(pd.Series([r.get(auc_col)])).iloc[0]
                rowm["Test_Acc_%"]  = _percentize(pd.Series([r.get(pref+"Test_Acc")])).iloc[0] if (pref+"Test_Acc") in df.columns else np.nan
                rowm["F1_%"]        = _percentize(pd.Series([r.get(pref+"F1")])).iloc[0]       if (pref+"F1") in df.columns else np.nan
                # prefer model-prefixed extras if present in the wide CSV
                for c in list(extras_present):
                    mc = pref + c if not c.startswith(pref) else c
                    if mc in df.columns:
                        rowm[c] = r.get(mc, np.nan)
                rows.append(rowm)

    out = pd.DataFrame(rows)
    for c in ["Feature_Map","Entanglement","Reps","Scaling"]:
        if c not in out.columns:
            out[c] = np.nan
    return out

# ============== classical RFE CSV parsing ==============

def _classical_top7_from_rfe_csv(path):
    """
    Try to re-find the best 7 classical features from a flexible 'classical_rfe_fixed_counts*.csv'.

    Expected possibilities:
      - A row per feature with a column like 'count_k7'/'selected_count_k7' (higher is better).
      - Or a column with 'score_k7' (higher is better).
      - Or a 'rank_k7' / 'avg_rank_k7' (lower is better).
      - Or a 'k' column with k==7 and then 'count'/'score'/'rank' columns.

    Returns: list of 7 feature names if successful, else None.
    """
    try:
        df = pd.read_csv(path)
    except Exception:
        return None

    cols = [c.strip() for c in df.columns]
    df.columns = cols

    # feature column name variants
    feat_col = None
    for cand in ["feature", "Feature", "name", "Name", "Feature_Name"]:
        if cand in df.columns:
            feat_col = cand
            break
    if feat_col is None:
        # sometimes index might be feature names
        if df.index.name:
            feat_col = df.index.name
        else:
            return None

    dfx = df.copy()

    # If a 'k' column exists, filter to k==7
    if "k" in dfx.columns:
        kk = pd.to_numeric(dfx["k"], errors="coerce")
        dfx = dfx[kk.eq(7)].copy()

    # Look for the best metric column for k=7
    metric_col = None
    metric_kind = None  # "desc" for counts/scores (bigger better); "asc" for ranks
    # Count/score columns with k7 in name
    for pat in ["count", "selected", "score"]:
        cand = [c for c in dfx.columns if ("7" in c and pat in c.lower())]
        if cand:
            # pick the first; you can refine if needed
            metric_col = cand[0]
            metric_kind = "desc"
            break
    # Rank columns with k7 in name (lower is better)
    if metric_col is None:
        for pat in ["rank", "avg_rank", "mean_rank"]:
            cand = [c for c in dfx.columns if ("7" in c and pat in c.lower())]
            if cand:
                metric_col = cand[0]
                metric_kind = "asc"
                break
    # Generic fallback if only one numeric metric present
    if metric_col is None:
        numeric_cols = [c for c in dfx.columns if c != feat_col and pd.api.types.is_numeric_dtype(dfx[c])]
        if len(numeric_cols) == 1:
            metric_col = numeric_cols[0]
            metric_kind = "desc"

    if metric_col is None:
        return None

    dfx = dfx[[feat_col, metric_col]].copy()
    order = True if metric_kind == "asc" else False
    dfx = dfx.sort_values(metric_col, ascending=order, kind="mergesort")
    top7 = dfx[feat_col].head(7).tolist()
    return top7 if len(top7) == 7 else None

# ============== main: TOP-10 + targeted config + classical link ==============

def find_features_for_config(quantum_csv="quantum_each_results.csv",
                             classical_csv="classical_rfe_fixed_counts.csv",
                             model="QSVC",
                             feature_map="z_feature_map",
                             entanglement="linear",
                             reps=1,
                             scaling_like="normaliz",  # matches "Normalization", "Normalize", "MinMax", etc.
                             top=10,
                             load_estimator=False):
    """
    1) Print TOP-10 @ k=7 (AUC desc → Acc desc → F1 desc) with full configs.
    2) Target config (model/feature_map/entanglement/reps/scaling_like) and recover its 7-feature set:
         - from champion row, else from matched LR row, else from best LR @ k=7.
    3) Link to classical RFE CSV to re-find the classical top-7 and show overlap.
    """
    pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

    # --- load & normalize ---
    qdf = pd.read_csv(quantum_csv)
    long = _to_long(qdf)

    # --- keep k=7 ---
    nf = pd.to_numeric(long["n_features"], errors="coerce")
    k7 = long[nf.eq(7)].copy()
    if k7.empty:
        print("No rows found for k=7 after scaling filter.")
        return

    # --- TOP-10 overall (aggregated unique configs) ---
    grp = (k7.groupby(["Model","Feature_Map","Entanglement","Reps","Scaling"], dropna=False)
             [["AUC_%","Test_Acc_%","F1_%"]].mean().reset_index())
    ranked = grp.sort_values(["AUC_%","Test_Acc_%","F1_%"],
                             ascending=[False, False, False],
                             kind="mergesort")
    cols_show = ["Model","Feature_Map","Entanglement","Reps","Scaling","AUC_%","Test_Acc_%","F1_%"]
    print("\nTOP-10 configs @ k=7 (AUC desc → Acc → F1):")
    print(ranked[cols_show].head(top).to_string(index=False))

    # --- target your specific config ---
    sl = k7["Scaling"].astype(str).str.lower()
    fm = k7["Feature_Map"].astype(str).str.strip().str.lower()
    ent = k7["Entanglement"].astype(str).str.strip().str.lower()
    reps_num = pd.to_numeric(k7["Reps"], errors="coerce")

    mask = (
        (k7["Model"] == model) &
        (fm == str(feature_map).lower()) &
        (ent == str(entanglement).lower()) &
        (reps_num.eq(float(reps))) &
        (sl.str.contains(str(scaling_like).lower()))
    )
    cand = k7[mask].copy()

    if cand.empty:
        print(f"\nNo rows match: Model={model}, Feature_Map={feature_map}, Entanglement={entanglement}, Reps={reps}, Scaling~{scaling_like}")
        print("Tip: check exact strings in your CSV (case/spacing) or adjust 'scaling_like'.")
        return

    cand = cand.sort_values(["AUC_%","Test_Acc_%","F1_%"], ascending=[False, False, False], kind="mergesort")
    champ = cand.iloc[0]
    print("\nTargeted champion row (exact pre-aggregation match):")
    print(champ.to_string())

    # --- try getting features directly from champion row (QSVC_* or global Selected_*) ---
    feat_info = _extract_feature_info(champ, model=model)
    features_from_qsvc = feat_info.get("selected_names") or feat_info.get("selected_idx")
    if features_from_qsvc:
        print("\nRecovered features directly from QSVC row:")
        print(" • Names:", feat_info.get("selected_names"))
        print(" • Indices:", feat_info.get("selected_idx"))
    else:
        print("\nNo Selected_* on QSVC row; trying to recover from LR…")

        # try to match an LR row using optional run keys (Feature_Set_ID, Fold, Seed, …)
        extras = _match_extra_keys(k7)
        lr = k7[k7["Model"] == "LR"].copy()
        lr_mask = pd.Series(True, index=lr.index)
        matched_any = False
        for ek in extras:
            if ek in champ.index and pd.notna(champ[ek]):
                lr_mask &= (lr[ek] == champ[ek])
                matched_any = True
        # if nothing to match on, keep LR rows with same scaling (likely consistent)
        if not matched_any and "Scaling" in lr.columns and pd.notna(champ.get("Scaling", np.nan)):
            lr_mask &= (lr["Scaling"] == champ["Scaling"])

        lr_cand = lr[lr_mask].copy()
        if lr_cand.empty:
            lr_cand = lr.copy()

        lr_cand = lr_cand.sort_values(["AUC_%","Test_Acc_%","F1_%"], ascending=[False, False, False], kind="mergesort")
        lr_best = lr_cand.iloc[0]
        print("\nMatched LR row used for classical features:")
        print(lr_best.to_string())

        feat_info = _extract_feature_info(lr_best, model="LR")
        if feat_info.get("selected_names") or feat_info.get("selected_idx"):
            print("\nRecovered features from LR row:")
            print(" • Names:", feat_info.get("selected_names"))
            print(" • Indices:", feat_info.get("selected_idx"))
        else:
            print("\nNo LR Selected_* found; will consult classical RFE CSV if provided.")

    # --- optionally load LR estimator for summary ---
    est_info = None
    try:
        lr_best  # might not exist if we found features from QSVC directly
        est_info = _extract_estimator_info(lr_best, model="LR", load_estimator=load_estimator)
        if est_info.get("estimator_source"):
            print(f"\nEstimator pointer: {est_info['estimator_source']} → {est_info['estimator_path_or_blob']}")
            if est_info.get("loaded"):
                print("Loaded LR estimator summary:", est_info.get("summary"))
    except NameError:
        pass

    selected_names = feat_info.get("selected_names")
    selected_idx   = feat_info.get("selected_idx")

    # --- link with classical RFE CSV to (re)find 7 best classical features ---
    classical_top7 = _classical_top7_from_rfe_csv(classical_csv) if classical_csv else None
    if classical_top7:
        print("\nTop-7 from classical RFE CSV (auto-detected):")
        print(" •", classical_top7)

        if selected_names:
            inter = [f for f in selected_names if f in classical_top7]
            print(f"\nOverlap (QSVC/LR-selected vs classical RFE): {len(inter)}/7")
            print(" •", inter)
        elif selected_idx:
            # If we only have indices, we can’t compare by name unless CSV provides full names
            print("\nOnly indices recovered; name-level overlap cannot be computed without full name map.")
    else:
        print("\nCould not auto-detect top-7 from classical RFE CSV (or file missing).")
        print("Tip: ensure it has columns like 'feature' + 'count_k7' or 'score_k7' (higher better),")
        print("     or 'rank_k7' / 'avg_rank_k7' (lower better), or a 'k' column with k==7.")

    # --- final: export a small manifest to help you reuse the exact set for simulations ---
    manifest = {
        "model": model,
        "feature_map": feature_map,
        "entanglement": entanglement,
        "reps": int(reps),
        "scaling_like": scaling_like,
        "k": 7,
        "selected_names": selected_names if selected_names else None,
        "selected_idx": selected_idx if selected_idx else None,
        "classical_top7": classical_top7 if classical_top7 else None
    }
    out = pd.DataFrame([manifest])
    out.to_csv("k7_qsvc_z_linear_r1_selected_features_manifest.csv", index=False)
    print("\nSaved: k7_qsvc_z_linear_r1_selected_features_manifest.csv")

# ---- run ----
# Adjust paths if needed.
find_features_for_config(
    quantum_csv="quantum_each_results.csv",
    classical_csv="classical_rfe_fixed_counts.csv",   # put your exact file name here
    model="QSVC",
    feature_map="z_feature_map",
    entanglement="linear",
    reps=1,
    scaling_like="normaliz",   # matches "Normalization", "Normalize", etc.
    top=10,
    load_estimator=False
)
